In [1]:
# Import Packages
import pandas as pd
import re

# Import from local files
from data_cleaning import DataCleaner
from extraction import extract_data
from entity_resolution import EntityResolver

resolver = EntityResolver()

# Set pandas display options
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)


# ----------------------------------------------------------
# Extract
# ----------------------------------------------------------

extracted_df = extract_data(
    "2025 Roll - Parcels Export.xlsx",
    "Only Delinquent Parcels"
)


# ----------------------------------------------------------
# Clean
# ----------------------------------------------------------

cleaner = DataCleaner()

clean_df = cleaner.clean_data(
    extracted_df
)


# ----------------------------------------------------------
# Entity Resolution
# ----------------------------------------------------------

(
    entity_occurrences,
    entities,
    parcel_entity_roles,
    association_evidence,
    entity_associations
) = EntityResolver().build(
    clean_df
)

# Create audit dataframe.
resolution_audit = (
    entity_occurrences
    .merge(
        entities[
            [
                "entity_id",
                "canonical_name",
                "canonical_address",
                "entity_type",
                "occurrence_count"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one",
        suffixes=("_source", "_final")
    )
)

resolution_audit.head()

,occurrence_id,parcel_number,entity_name,associated_address,role,source_entity_name,source_column,source_occurrence_id,owner_line_continuation_applied,owner_line_continuation_rule,entity_type_source,joint_parse_rule,joint_parse_confidence,joint_expanded,joint_party_index,joint_party_count,organization_type,organization_classification_confidence,address_source,entity_id,canonical_name,canonical_address,entity_type_final,occurrence_count
0,1,01-01-201-013,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Owner,POLOMCAK CLAUDIA,owner_name,1,False,,Person,,,False,NaN,NaN,NaN,NaN,owner_address,3587,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Person,2
1,2,01-01-251-013,CLAWSON DORRINDA &,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA &,owner_name,2,False,,Joint,,,False,NaN,NaN,NaN,NaN,owner_address,1008,CLAWSON DORRINDA &,"10591 N 12TH ST, PLAINWELL, MI 49080",Joint,2
2,3,01-01-251-050,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA L,owner_name,3,False,,Person,,,False,NaN,NaN,NaN,NaN,owner_address,1009,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,6
3,4,01-01-251-060,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA L,owner_name,4,False,,Person,,,False,NaN,NaN,NaN,NaN,owner_address,1009,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,6
4,5,01-01-301-050,"SPIEKERMANN, TIM","10238 N 10TH ST, PLAINWELL, MI 49080",Owner,SPIEKERMANN TIM & JUDY,owner_name,5,False,,Person,shared_surname_single_right_name,High,True,1.0,2.0,NaN,NaN,owner_address,4260,"SPIEKERMANN, TIM","10238 N 10TH ST, PLAINWELL, MI 49080",Person,2


In [2]:
entities["occurrence_count"].sum()

np.int64(12056)

In [3]:
# ==========================================================
# OWNER_NAME / OWNER_NAME_2 CONTINUATION-LINE AUDIT
# ==========================================================

def owner_line_audit_key(value):
    """
    Create a comparison-only representation for detecting
    possible source-line continuations.

    This does not modify production entity names.
    """

    if pd.isna(value):
        return ""

    value = str(value).upper().strip()

    value = re.sub(
        r"[^A-Z0-9]+",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()

In [4]:
name_columns = [
    "owner_name",
    "owner_name_2",
    "owner_care_of",
    "taxpayer_name",
    "taxpayer_care_of",
    "tax_send_to_name"
]

known_source_name_keys = set()

for column in name_columns:

    if column not in clean_df.columns:
        continue

    known_source_name_keys.update(
        clean_df[column]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[
            lambda x:
                x != ""
        ]
        .map(
            owner_line_audit_key
        )
    )

known_source_name_keys.discard("")

In [5]:
owner_line_pairs = (
    clean_df[
        (
            clean_df[
                "owner_name"
            ].notna()
        )
        &
        (
            clean_df[
                "owner_name"
            ].str.strip()
            != ""
        )
        &
        (
            clean_df[
                "owner_name_2"
            ].notna()
        )
        &
        (
            clean_df[
                "owner_name_2"
            ].str.strip()
            != ""
        )
    ]
    [
        [
            "parcel_number",
            "owner_name",
            "owner_name_2",
            "owner_address"
        ]
    ]
    .drop_duplicates()
    .copy()
)

In [6]:
owner_line_pairs[
    "forward_combined_name"
] = (
    owner_line_pairs[
        "owner_name"
    ].str.strip()
    +
    " "
    +
    owner_line_pairs[
        "owner_name_2"
    ].str.strip()
)

owner_line_pairs[
    "reverse_combined_name"
] = (
    owner_line_pairs[
        "owner_name_2"
    ].str.strip()
    +
    " "
    +
    owner_line_pairs[
        "owner_name"
    ].str.strip()
)

owner_line_pairs[
    "forward_key"
] = (
    owner_line_pairs[
        "forward_combined_name"
    ]
    .map(
        owner_line_audit_key
    )
)

owner_line_pairs[
    "reverse_key"
] = (
    owner_line_pairs[
        "reverse_combined_name"
    ]
    .map(
        owner_line_audit_key
    )
)

In [7]:
generic_continuation_pattern = re.compile(
    r"""
    ^
    (?:
        TRUST
        |
        LIVING\s+TRUST
        |
        REVOCABLE\s+TRUST
        |
        REVOCABLE\s+TRST
        |
        LIVING\s+TRST
        |
        ESTATE\s+OF
    )
    $
    """,
    flags=re.IGNORECASE | re.VERBOSE
)


def continuation_reasons(row):

    reasons = []

    owner_name = str(
        row["owner_name"]
    ).strip()

    owner_name_2 = str(
        row["owner_name_2"]
    ).strip()

    # Does combining the two lines reproduce a complete
    # source name that occurs elsewhere in the data?
    if (
        row["forward_key"]
        in known_source_name_keys
    ):
        reasons.append(
            "FORWARD_COMBINATION_EXISTS_ELSEWHERE"
        )

    if (
        row["reverse_key"]
        in known_source_name_keys
    ):
        reasons.append(
            "REVERSE_COMBINATION_EXISTS_ELSEWHERE"
        )

    # Examples:
    # KALAMAZOO EAST CONGREGATION OF
    # JEHOVAH'S WITNESS
    if re.search(
        r"\bOF$",
        owner_name,
        flags=re.IGNORECASE
    ):
        reasons.append(
            "OWNER_NAME_ENDS_WITH_OF"
        )

    # Examples:
    # DECKER, LYDIA
    # LIVING TRUST
    #
    # APPLEGATE, NEIL E & SHARON K
    # REVOCABLE TRUST
    if generic_continuation_pattern.fullmatch(
        owner_name_2
    ):
        reasons.append(
            "OWNER_NAME_2_GENERIC_CONTINUATION"
        )

    # Example:
    # WILLIAMS BRIAN R (DECEASED)
    # ESTATE OF
    if generic_continuation_pattern.fullmatch(
        owner_name
    ):
        reasons.append(
            "OWNER_NAME_GENERIC_CONTINUATION"
        )

    # Example:
    # STAUFFER JENNIFER
    # TRUSTEE OF THE BELL FAMILY TRUST
    if re.match(
        r"^TRUSTEE\s+OF\b",
        owner_name_2,
        flags=re.IGNORECASE
    ):
        reasons.append(
            "OWNER_NAME_2_STARTS_TRUSTEE_OF"
        )

    return tuple(reasons)

In [8]:
owner_line_pairs[
    "continuation_reasons"
] = (
    owner_line_pairs
    .apply(
        continuation_reasons,
        axis=1
    )
)

owner_line_continuation_candidates = (
    owner_line_pairs[
        owner_line_pairs[
            "continuation_reasons"
        ].map(bool)
    ]
    .copy()
    .sort_values(
        [
            "owner_name",
            "owner_name_2",
            "parcel_number"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

print(
    "Owner-name pairs with both lines populated:",
    len(owner_line_pairs)
)

print(
    "Potential continuation-line pairs:",
    len(
        owner_line_continuation_candidates
    )
)

owner_line_continuation_candidates

Owner-name pairs with both lines populated: 217
Potential continuation-line pairs: 13


,parcel_number,owner_name,owner_name_2,owner_address,forward_combined_name,reverse_combined_name,forward_key,reverse_key,continuation_reasons
0,06-16-349-005,"APPLEGATE, NEIL E & SHARON K",REVOCABLE TRUST,"4912 N DR S, E LEROY, MI 49051","APPLEGATE, NEIL E & SHARON K REVOCABLE TRUST","REVOCABLE TRUST APPLEGATE, NEIL E & SHARON K",APPLEGATE NEIL E SHARON K REVOCABLE TRUST,REVOCABLE TRUST APPLEGATE NEIL E SHARON K,"(OWNER_NAME_2_GENERIC_CONTINUATION,)"
1,06-09-477-002,"DARDEN, MURPHY SR ESTATE OF","DARDEN MURPHY JR, DARDEN VICENT B","628 LULU ST, KALAMAZOO, MI 49007","DARDEN, MURPHY SR ESTATE OF DARDEN MURPHY JR, ...","DARDEN MURPHY JR, DARDEN VICENT B DARDEN, MURP...",DARDEN MURPHY SR ESTATE OF DARDEN MURPHY JR DA...,DARDEN MURPHY JR DARDEN VICENT B DARDEN MURPHY...,"(OWNER_NAME_ENDS_WITH_OF,)"
2,06-35-147-001,"DECKER, LYDIA",LIVING TRUST,"3301 PORTAGE ST, KALAMAZOO, MI 49001","DECKER, LYDIA LIVING TRUST","LIVING TRUST DECKER, LYDIA",DECKER LYDIA LIVING TRUST,LIVING TRUST DECKER LYDIA,"(OWNER_NAME_2_GENERIC_CONTINUATION,)"
3,06-35-263-182,"DECKER, LYDIA",LIVING TRUST,"1828 MILES AVE, KALAMAZOO, MI 49001","DECKER, LYDIA LIVING TRUST","LIVING TRUST DECKER, LYDIA",DECKER LYDIA LIVING TRUST,LIVING TRUST DECKER LYDIA,"(OWNER_NAME_2_GENERIC_CONTINUATION,)"
4,06-29-437-025,ELIZABETH A SCHUBERT RECOVERABLE,LIVING TRUST,"1822 CHEVY CHASE BLVD, KALAMAZOO, MI 49008",ELIZABETH A SCHUBERT RECOVERABLE LIVING TRUST,LIVING TRUST ELIZABETH A SCHUBERT RECOVERABLE,ELIZABETH A SCHUBERT RECOVERABLE LIVING TRUST,LIVING TRUST ELIZABETH A SCHUBERT RECOVERABLE,"(OWNER_NAME_2_GENERIC_CONTINUATION,)"
5,06-01-205-031,KALAMAZOO EAST CONGREGATION OF,JEHOVAH'S WITNESS,"3414 ENTERPRISE, KALAMAZOO, MI 49004",KALAMAZOO EAST CONGREGATION OF JEHOVAH'S WITNESS,JEHOVAH'S WITNESS KALAMAZOO EAST CONGREGATION OF,KALAMAZOO EAST CONGREGATION OF JEHOVAH S WITNESS,JEHOVAH S WITNESS KALAMAZOO EAST CONGREGATION OF,"(OWNER_NAME_ENDS_WITH_OF,)"
6,06-35-175-472,"LAGO, JANE M TTEE OF",JANE M LAGO REVOCABLE TRUST,"1227 ROYCE AVE, KALAMAZOO, MI 49001","LAGO, JANE M TTEE OF JANE M LAGO REVOCABLE TRUST","JANE M LAGO REVOCABLE TRUST LAGO, JANE M TTEE OF",LAGO JANE M TTEE OF JANE M LAGO REVOCABLE TRUST,JANE M LAGO REVOCABLE TRUST LAGO JANE M TTEE OF,"(OWNER_NAME_ENDS_WITH_OF,)"
7,06-14-150-003,"MCKINLEY, DEON",MVP AUTO SALES,"717 RIVERVIEW DR, KALAMAZOO, MI 49048","MCKINLEY, DEON MVP AUTO SALES","MVP AUTO SALES MCKINLEY, DEON",MCKINLEY DEON MVP AUTO SALES,MVP AUTO SALES MCKINLEY DEON,"(FORWARD_COMBINATION_EXISTS_ELSEWHERE,)"
8,06-23-355-316,"MUNYE, MALEYSIA","ALEMAYEHU, ESKINDER","1110 60TH ST SE, KENTWOOD, MI 49508","MUNYE, MALEYSIA ALEMAYEHU, ESKINDER","ALEMAYEHU, ESKINDER MUNYE, MALEYSIA",MUNYE MALEYSIA ALEMAYEHU ESKINDER,ALEMAYEHU ESKINDER MUNYE MALEYSIA,"(FORWARD_COMBINATION_EXISTS_ELSEWHERE,)"
9,01-35-170-220,NIEWOONDER ED & KAY,TRUST,"6817 COUNTRY VIEW, KALAMAZOO, MI 49009",NIEWOONDER ED & KAY TRUST,TRUST NIEWOONDER ED & KAY,NIEWOONDER ED KAY TRUST,TRUST NIEWOONDER ED KAY,"(OWNER_NAME_2_GENERIC_CONTINUATION,)"


In [9]:
continuation_reason_counts = (
    owner_line_continuation_candidates[
        "continuation_reasons"
    ]
    .explode()
    .value_counts()
)

continuation_reason_counts

continuation_reasons
OWNER_NAME_2_GENERIC_CONTINUATION       6
OWNER_NAME_ENDS_WITH_OF                 4
FORWARD_COMBINATION_EXISTS_ELSEWHERE    2
OWNER_NAME_2_STARTS_TRUSTEE_OF          1
Name: count, dtype: int64

In [10]:
owner_line_continuation_candidates.to_csv(
    "owner_line_continuation_candidates.csv",
    index=False
)

## Repeated surname audit

In [11]:
def has_repeated_surname_comma_structure(name):
    """
    Flag names that appear to contain two full person names
    separated by a comma, where both sides begin with the
    same surname.

    Example
    -------
    DARDEN MURPHY JR, DARDEN VICENT B
        -> True

    SMITH, JOHN A
        -> False
    """

    if pd.isna(name):
        return False

    name = str(name).upper().strip()

    # This audit is specifically for a two-part comma structure.
    parts = [
        part.strip()
        for part in name.split(",")
        if part.strip()
    ]

    if len(parts) != 2:
        return False

    left_tokens = re.findall(
        r"[A-Z]+(?:['-][A-Z]+)*",
        parts[0]
    )

    right_tokens = re.findall(
        r"[A-Z]+(?:['-][A-Z]+)*",
        parts[1]
    )

    # Each side should look substantial enough to represent
    # a surname plus at least one given-name token.
    if (
            len(left_tokens) < 2
            or len(right_tokens) < 2
    ):
        return False

    return left_tokens[0] == right_tokens[0]

In [12]:
repeating_surname_comma_audit = (
    entity_occurrences[
        (
            entity_occurrences["entity_type"]
            == "Person"
        )
        &
        (
            entity_occurrences["entity_name"]
            .apply(
                has_repeated_surname_comma_structure
            )
        )
    ]
    [
        [
            "entity_id",
            "source_occurrence_id",
            "source_entity_name",
            "entity_name",
            "parcel_number",
            "role",
            "associated_address",
            "source_column"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "entity_name",
            "parcel_number",
            "role"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Person occurrences with a repeated-surname "
    "comma structure:",
    len(repeating_surname_comma_audit)
)

print(
    "Distinct entity names:",
    repeating_surname_comma_audit[
        "entity_name"
    ].nunique()
)

repeating_surname_comma_audit

Person occurrences with a repeated-surname comma structure: 0
Distinct entity names: 0


,entity_id,source_occurrence_id,source_entity_name,entity_name,parcel_number,role,associated_address,source_column


In [13]:
repeating_surname_comma_summary = (
    repeating_surname_comma_audit
    .groupby(
        [
            "entity_id",
            "entity_name",
            "associated_address"
        ],
        dropna=False
    )
    .agg(
        occurrence_count=(
            "source_occurrence_id",
            "nunique"
        ),
        parcel_count=(
            "parcel_number",
            "nunique"
        ),
        parcels=(
            "parcel_number",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            x.dropna().astype(str)
                        )
                    )
                )
        ),
        roles=(
            "role",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            x.dropna().astype(str)
                        )
                    )
                )
        ),
        source_names=(
            "source_entity_name",
            lambda x:
                " | ".join(
                    sorted(
                        set(
                            x.dropna().astype(str)
                        )
                    )
                )
        )
    )
    .reset_index()
    .sort_values(
        "entity_name"
    )
    .reset_index(drop=True)
)

print(
    "Distinct suspicious entities:",
    len(repeating_surname_comma_summary)
)

repeating_surname_comma_summary

Distinct suspicious entities: 0


,entity_id,entity_name,associated_address,occurrence_count,parcel_count,parcels,roles,source_names


In [14]:
darden_joint_check = (
    entity_occurrences[
        entity_occurrences[
            "source_entity_name"
        ].isin(
            [
                "DARDEN MURPHY JR, DARDEN VICENT B",
                "DARDEN RICKIE L, DARDEN VANASSA A"
            ]
        )
    ]
    [
        [
            "source_entity_name",
            "entity_name",
            "entity_type",
            "parcel_number",
            "role",
            "associated_address"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "source_entity_name",
            "role"
        ]
    )
    .reset_index(drop=True)
)

darden_joint_check

,source_entity_name,entity_name,entity_type,parcel_number,role,associated_address
0,"DARDEN MURPHY JR, DARDEN VICENT B","DARDEN MURPHY JR, DARDEN VICENT B",Joint,06-09-477-002,Owner,"628 LULU ST, KALAMAZOO, MI 49007"
1,"DARDEN RICKIE L, DARDEN VANASSA A","DARDEN RICKIE L, DARDEN VANASSA A",Joint,06-09-477-002,Owner Care Of,"628 LULU ST, KALAMAZOO, MI 49007"
2,"DARDEN RICKIE L, DARDEN VANASSA A","DARDEN RICKIE L, DARDEN VANASSA A",Joint,06-09-477-002,Taxpayer Care Of,"628 LULU ST, KALAMAZOO, MI 49007"


In [15]:
print(
    "Occurrence count:",
    entities["occurrence_count"].sum()
)

Occurrence count: 12056


In [16]:
repeating_surname_comma_audit = (
    entity_occurrences[
        (
            entity_occurrences["entity_type"]
            == "Person"
        )
        &
        (
            entity_occurrences["entity_name"]
            .apply(
                has_repeated_surname_comma_structure
            )
        )
    ]
)

print(
    "Remaining repeated-surname comma Persons:",
    len(repeating_surname_comma_audit)
)

Remaining repeated-surname comma Persons: 0


## Association evidence audit

In [17]:
print(
    "Association evidence rows:",
    len(association_evidence)
)

association_evidence[
    "evidence_strength"
].value_counts()

Association evidence rows: 491


evidence_strength
Medium    264
High      183
Low        44
Name: count, dtype: int64

In [18]:
association_evidence[
    "evidence_rule"
].value_counts()

evidence_rule
SHARED_OWNER_ADDRESS                          97
SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER         79
SHARED_TAXPAYER_ADDRESS                       69
SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER         55
SAME_PARCEL_CO_OWNER                          49
SAME_PARCEL_ORG_OWNER_PERSON_TAX_SEND_TO      49
SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER      49
SAME_PARCEL_ORG_OWNER_PERSON_OWNER_CARE_OF    44
Name: count, dtype: int64

In [19]:
print(
    "Duplicate evidence rows:",
    association_evidence.duplicated(
        subset=[
            "entity_id_a",
            "entity_id_b",
            "evidence_rule",
            "parcel_number_a",
            "parcel_number_b",
            "role_a",
            "role_b"
        ]
    ).sum()
)

Duplicate evidence rows: 0


In [20]:
print(
    "Total association evidence rows:",
    len(association_evidence)
)

print("\nEvidence rules:")
print(
    association_evidence[
        "evidence_rule"
    ].value_counts()
)

print("\nEvidence strengths:")
print(
    association_evidence[
        "evidence_strength"
    ].value_counts()
)

Total association evidence rows: 491

Evidence rules:
evidence_rule
SHARED_OWNER_ADDRESS                          97
SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER         79
SHARED_TAXPAYER_ADDRESS                       69
SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER         55
SAME_PARCEL_CO_OWNER                          49
SAME_PARCEL_ORG_OWNER_PERSON_TAX_SEND_TO      49
SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER      49
SAME_PARCEL_ORG_OWNER_PERSON_OWNER_CARE_OF    44
Name: count, dtype: int64

Evidence strengths:
evidence_strength
Medium    264
High      183
Low        44
Name: count, dtype: int64


In [21]:
shared_owner_evidence = (
    association_evidence[
        association_evidence[
            "evidence_rule"
        ]
        == "SHARED_OWNER_ADDRESS"
    ]
)

print(
    "Shared-owner-address evidence rows:",
    len(shared_owner_evidence)
)

print(
    "Distinct entity pairs:",
    shared_owner_evidence[
        [
            "entity_id_a",
            "entity_id_b"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct shared addresses:",
    shared_owner_evidence[
        "shared_address"
    ].nunique()
)

print(
    "Duplicate shared-owner evidence:",
    shared_owner_evidence.duplicated(
        subset=[
            "entity_id_a",
            "entity_id_b",
            "evidence_rule",
            "shared_address"
        ]
    ).sum()
)

Shared-owner-address evidence rows: 97
Distinct entity pairs: 97
Distinct shared addresses: 52
Duplicate shared-owner evidence: 0


In [22]:
print(
    "Shared-owner rows with parcel_number_a populated:",
    shared_owner_evidence[
        "parcel_number_a"
    ].notna().sum()
)

print(
    "Shared-owner rows with parcel_number_b populated:",
    shared_owner_evidence[
        "parcel_number_b"
    ].notna().sum()
)

Shared-owner rows with parcel_number_a populated: 0
Shared-owner rows with parcel_number_b populated: 0


In [23]:
# Every evidence ID should be unique.
assert association_evidence[
    "evidence_id"
].is_unique

# Entity pairs should always be normalized.
assert (
    association_evidence[
        "entity_id_a"
    ]
    <
    association_evidence[
        "entity_id_b"
    ]
).all()

# Every referenced entity should actually exist.
valid_entity_ids = set(
    entities["entity_id"]
)

assert set(
    association_evidence["entity_id_a"]
).issubset(valid_entity_ids)

assert set(
    association_evidence["entity_id_b"]
).issubset(valid_entity_ids)

# Shared-owner-address evidence should have exactly
# the expected characteristics.
shared_owner = association_evidence[
    association_evidence["evidence_rule"]
    == "SHARED_OWNER_ADDRESS"
]

assert (
    shared_owner["evidence_strength"]
    == "Medium"
).all()

assert (
    shared_owner["address_type"]
    == "Owner"
).all()

assert (
    shared_owner["role_a"]
    == "Owner"
).all()

assert (
    shared_owner["role_b"]
    == "Owner"
).all()

assert shared_owner[
    "shared_address"
].notna().all()

assert (
    shared_owner[
        "shared_address"
    ].str.strip()
    != ""
).all()

print("Stage 5 shared-owner-address validation passed.")

Stage 5 shared-owner-address validation passed.


## Shared taxpayer address audit

In [24]:
# ----------------------------------------------------------
# One taxpayer address per parcel
# ----------------------------------------------------------

parcel_taxpayer_addresses = (
    clean_df[
        [
            "parcel_number",
            "taxpayer_address"
        ]
    ]
    .copy()
)

parcel_taxpayer_addresses = (
    parcel_taxpayer_addresses[
        (
            parcel_taxpayer_addresses[
                "taxpayer_address"
            ].notna()
        )
        &
        (
            parcel_taxpayer_addresses[
                "taxpayer_address"
            ].str.strip()
            != ""
        )
    ]
    .drop_duplicates()
)

In [25]:
taxpayer_address_counts = (
    parcel_taxpayer_addresses
    .groupby("parcel_number")[
        "taxpayer_address"
    ]
    .nunique()
)

print(
    "Parcels with more than one taxpayer address:",
    (taxpayer_address_counts > 1).sum()
)

Parcels with more than one taxpayer address: 0


In [26]:
owner_entities_by_parcel = (
    parcel_entity_roles[
        parcel_entity_roles[
            "role"
        ] == "Owner"
    ]
    [
        [
            "parcel_number",
            "entity_id"
        ]
    ]
    .drop_duplicates()
)

owner_taxpayer_address_entities = (
    owner_entities_by_parcel
    .merge(
        parcel_taxpayer_addresses,
        on="parcel_number",
        how="inner",
        validate="many_to_one"
    )
    .merge(
        entities[
            [
                "entity_id",
                "entity_type",
                "canonical_name"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one"
    )
)

In [27]:
print(
    "Owner rows with missing entity type:",
    owner_taxpayer_address_entities[
        "entity_type"
    ].isna().sum()
)

Owner rows with missing entity type: 0


### Profile taxpayer addresses

In [28]:
taxpayer_address_profile = (
    owner_taxpayer_address_entities
    .groupby(
        "taxpayer_address"
    )
    .agg(
        distinct_entities=(
            "entity_id",
            "nunique"
        ),
        distinct_parcels=(
            "parcel_number",
            "nunique"
        )
    )
    .reset_index()
)

In [29]:
person_counts = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Person"
    ]
    .groupby(
        "taxpayer_address"
    )[
        "entity_id"
    ]
    .nunique()
    .rename(
        "distinct_person_entities"
    )
)

organization_counts = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Organization"
    ]
    .groupby(
        "taxpayer_address"
    )[
        "entity_id"
    ]
    .nunique()
    .rename(
        "distinct_organization_entities"
    )
)

taxpayer_address_profile = (
    taxpayer_address_profile
    .merge(
        person_counts,
        on="taxpayer_address",
        how="left"
    )
    .merge(
        organization_counts,
        on="taxpayer_address",
        how="left"
    )
    .fillna(
        {
            "distinct_person_entities": 0,
            "distinct_organization_entities": 0
        }
    )
)

taxpayer_address_profile[
    [
        "distinct_person_entities",
        "distinct_organization_entities"
    ]
] = (
    taxpayer_address_profile[
        [
            "distinct_person_entities",
            "distinct_organization_entities"
        ]
    ]
    .astype(int)
)

taxpayer_address_profile = (
    taxpayer_address_profile
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

In [30]:
print(
    "Taxpayer addresses:",
    len(taxpayer_address_profile)
)

print(
    "Addresses linked to >1 owner entity:",
    (
        taxpayer_address_profile[
            "distinct_entities"
        ] > 1
    ).sum()
)

print(
    "Addresses containing both Person and Organization owners:",
    (
        (
            taxpayer_address_profile[
                "distinct_person_entities"
            ] > 0
        )
        &
        (
            taxpayer_address_profile[
                "distinct_organization_entities"
            ] > 0
        )
    ).sum()
)

taxpayer_address_profile[
    [
        "distinct_entities",
        "distinct_parcels"
    ]
].describe()

Taxpayer addresses: 1558
Addresses linked to >1 owner entity: 436
Addresses containing both Person and Organization owners: 50


,distinct_entities,distinct_parcels
count,1558.000000,1558.000000
mean,1.348524,1.269576
std,0.708582,1.058585
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,1.000000
max,14.000000,20.000000


In [31]:
taxpayer_address_persons = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Person"
    ]
    [
        [
            "parcel_number",
            "entity_id",
            "canonical_name",
            "taxpayer_address"
        ]
    ]
    .rename(
        columns={
            "parcel_number":
                "person_parcel",
            "entity_id":
                "person_id",
            "canonical_name":
                "person_name"
        }
    )
)

taxpayer_address_organizations = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Organization"
    ]
    [
        [
            "parcel_number",
            "entity_id",
            "canonical_name",
            "taxpayer_address"
        ]
    ]
    .rename(
        columns={
            "parcel_number":
                "organization_parcel",
            "entity_id":
                "organization_id",
            "canonical_name":
                "organization_name"
        }
    )
)

shared_taxpayer_address_candidates = (
    taxpayer_address_organizations
    .merge(
        taxpayer_address_persons,
        on="taxpayer_address",
        how="inner"
    )
)

shared_taxpayer_address_candidates = (
    shared_taxpayer_address_candidates[
        shared_taxpayer_address_candidates[
            "organization_parcel"
        ]
        !=
        shared_taxpayer_address_candidates[
            "person_parcel"
        ]
    ]
    .copy()
)

In [32]:
print(
    "Cross-parcel shared-taxpayer-address candidate rows:",
    len(shared_taxpayer_address_candidates)
)

print(
    "Distinct Person-Organization pairs:",
    shared_taxpayer_address_candidates[
        [
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct shared taxpayer addresses:",
    shared_taxpayer_address_candidates[
        "taxpayer_address"
    ].nunique()
)

print(
    "Distinct Person-Organization-address combinations:",
    shared_taxpayer_address_candidates[
        [
            "person_id",
            "organization_id",
            "taxpayer_address"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

Cross-parcel shared-taxpayer-address candidate rows: 150
Distinct Person-Organization pairs: 69
Distinct shared taxpayer addresses: 36
Distinct Person-Organization-address combinations: 69


In [33]:
candidate_taxpayer_addresses = set(
    shared_taxpayer_address_candidates[
        "taxpayer_address"
    ]
)

taxpayer_address_member_summary = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "taxpayer_address"
        ].isin(
            candidate_taxpayer_addresses
        )
    ]
    .groupby(
        [
            "taxpayer_address",
            "entity_id",
            "canonical_name",
            "entity_type"
        ],
        dropna=False
    )
    .agg(
        parcel_count=(
            "parcel_number",
            "nunique"
        ),
        parcels=(
            "parcel_number",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            x.dropna()
                            .astype(str)
                        )
                    )
                )
        )
    )
    .reset_index()
    .merge(
        taxpayer_address_profile[
            [
                "taxpayer_address",
                "distinct_entities",
                "distinct_parcels",
                "distinct_person_entities",
                "distinct_organization_entities"
            ]
        ],
        on="taxpayer_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels",
            "taxpayer_address",
            "entity_type",
            "canonical_name"
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

taxpayer_address_member_summary

,taxpayer_address,entity_id,canonical_name,entity_type,parcel_count,parcels,distinct_entities,distinct_parcels,distinct_person_entities,distinct_organization_entities
0,"218 GLADYS ST, PORTAGE, MI 49002",3031,MIKA FIRST LLC,Organization,3,"06-10-315-021, 06-10-389-037, 06-16-248-002",5,7,2,3
1,"218 GLADYS ST, PORTAGE, MI 49002",4213,SMS HOLDINGS LLC,Organization,1,06-10-377-011,5,7,2,3
2,"218 GLADYS ST, PORTAGE, MI 49002",4214,SMS MICHIGAN HOLDINGS LLC,Organization,1,06-09-397-007,5,7,2,3
3,"218 GLADYS ST, PORTAGE, MI 49002",629,"BRINKS, TERRY",Person,1,06-16-328-005,5,7,2,3
4,"218 GLADYS ST, PORTAGE, MI 49002",4476,"THOMAS, TIMOTHY",Person,1,06-02-320-010,5,7,2,3
...,...,...,...,...,...,...,...,...,...,...
98,"3131 BRYNMAWR DR, PORTAGE, MI 49024-5654",661,"BROOKS, RONALD",Person,1,06-10-398-036,2,2,1,1
99,"4205 AMELIA WAY, NAPLES, FL 34119-9043",4144,"SKARTSIARIS, THEODORE MARITAL TRST",Organization,1,06-34-365-001,2,2,1,1
100,"4205 AMELIA WAY, NAPLES, FL 34119-9043",4143,"SKARTSIARIS, THEODORE E, TRUSTEE",Person,1,06-15-354-130,2,2,1,1
101,"51842 WELCHER DR, MARCELLUS, MI 49067",1771,"GOVE, DOUGLAS ESTATE",Organization,1,06-15-127-001,2,2,1,1


In [34]:
taxpayer_address_member_summary.to_csv(
    "taxpayer_address_member_summary.csv",
    index=False
)

shared_taxpayer_address_candidates.to_csv(
    "shared_taxpayer_address_candidates.csv",
    index=False
)

In [35]:
print(
    "Parcels with more than one taxpayer address:",
    (taxpayer_address_counts > 1).sum()
)

print(
    "Owner rows with missing entity type:",
    owner_taxpayer_address_entities[
        "entity_type"
    ].isna().sum()
)

Parcels with more than one taxpayer address: 0
Owner rows with missing entity type: 0


In [36]:
shared_taxpayer_address_candidates

,organization_parcel,organization_id,organization_name,taxpayer_address,person_parcel,person_id,person_name
2,06-09-397-007,4214,SMS MICHIGAN HOLDINGS LLC,"218 GLADYS ST, PORTAGE, MI 49002",06-02-320-010,4476,"THOMAS, TIMOTHY"
3,06-09-397-007,4214,SMS MICHIGAN HOLDINGS LLC,"218 GLADYS ST, PORTAGE, MI 49002",06-16-328-005,629,"BRINKS, TERRY"
4,06-10-311-025,1365,"DREAM MANAGEMENT, LLC","1103 WOODWARD AVE, KALAMAZOO, MI 49007",06-10-380-023,181,"ANDERSON, DEJA"
5,06-10-315-021,3031,MIKA FIRST LLC,"218 GLADYS ST, PORTAGE, MI 49002",06-02-320-010,4476,"THOMAS, TIMOTHY"
6,06-10-315-021,3031,MIKA FIRST LLC,"218 GLADYS ST, PORTAGE, MI 49002",06-16-328-005,629,"BRINKS, TERRY"
...,...,...,...,...,...,...,...
174,07-24-215-160,4453,THE COPPER CREEK TRUST,"4960 S 36TH ST, CLIMAX, MI 49034",07-24-215-100,1786,"GRAHAM, WILLIAM D"
175,07-24-215-160,4453,THE COPPER CREEK TRUST,"4960 S 36TH ST, CLIMAX, MI 49034",07-24-215-110,1786,"GRAHAM, WILLIAM D"
177,08-18-301-140,2802,MARION CHARLES & BETTY TRUST,"12195 E MICHIGAN, GALESBURG, MI 49053",06-23-363-453,1868,"HAIST, MARCIA MAY"
178,08-18-301-140,2802,MARION CHARLES & BETTY TRUST,"12195 E MICHIGAN, GALESBURG, MI 49053",07-26-255-030,2804,"MARION, BETTY"


## Stage 6 pre-implementation audit

In [37]:
strength_rank = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}


# ----------------------------------------------------------
# Summarize all evidence at the entity-pair level
# ----------------------------------------------------------

pair_evidence_summary = (
    association_evidence
    .groupby(
        [
            "entity_id_a",
            "entity_id_b"
        ],
        dropna=False
    )
    .agg(
        evidence_count=(
            "evidence_id",
            "count"
        ),

        distinct_evidence_rules=(
            "evidence_rule",
            "nunique"
        ),

        evidence_rules=(
            "evidence_rule",
            lambda x:
                tuple(
                    sorted(
                        set(
                            x.dropna()
                        )
                    )
                )
        ),

        evidence_strengths=(
            "evidence_strength",
            lambda x:
                tuple(
                    sorted(
                        set(
                            x.dropna()
                        ),
                        key=lambda value:
                            strength_rank.get(
                                value,
                                0
                            ),
                        reverse=True
                    )
                )
        ),

        strongest_evidence_strength=(
            "evidence_strength",
            lambda x:
                max(
                    x.dropna(),
                    key=lambda value:
                        strength_rank.get(
                            value,
                            0
                        )
                )
        )
    )
    .reset_index()
)


# ----------------------------------------------------------
# Add readable entity names/types
# ----------------------------------------------------------

entity_a_lookup = (
    entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type"
        ]
    ]
    .rename(
        columns={
            "entity_id":
                "entity_id_a",
            "canonical_name":
                "entity_name_a",
            "entity_type":
                "entity_type_a"
        }
    )
)

entity_b_lookup = (
    entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type"
        ]
    ]
    .rename(
        columns={
            "entity_id":
                "entity_id_b",
            "canonical_name":
                "entity_name_b",
            "entity_type":
                "entity_type_b"
        }
    )
)

pair_evidence_summary = (
    pair_evidence_summary
    .merge(
        entity_a_lookup,
        on="entity_id_a",
        how="left",
        validate="many_to_one"
    )
    .merge(
        entity_b_lookup,
        on="entity_id_b",
        how="left",
        validate="many_to_one"
    )
)


# ----------------------------------------------------------
# Human-readable rule-combination label
# ----------------------------------------------------------

pair_evidence_summary[
    "rule_combination"
] = (
    pair_evidence_summary[
        "evidence_rules"
    ]
    .apply(
        lambda rules:
            " + ".join(rules)
    )
)

In [38]:
# ==========================================================
# OVERALL MULTIPLICITY
# ==========================================================

total_pairs = len(
    pair_evidence_summary
)

multiple_evidence_pairs = (
    pair_evidence_summary[
        pair_evidence_summary[
            "evidence_count"
        ] > 1
    ]
    .copy()
)

multiple_rule_pairs = (
    pair_evidence_summary[
        pair_evidence_summary[
            "distinct_evidence_rules"
        ] > 1
    ]
    .copy()
)

repeated_same_rule_pairs = (
    pair_evidence_summary[
        (
            pair_evidence_summary[
                "evidence_count"
            ] > 1
        )
        &
        (
            pair_evidence_summary[
                "distinct_evidence_rules"
            ] == 1
        )
    ]
    .copy()
)

print(
    "Total distinct entity pairs:",
    total_pairs
)

print(
    "Pairs with multiple evidence rows:",
    len(
        multiple_evidence_pairs
    )
)

print(
    "Pairs supported by multiple distinct rules:",
    len(
        multiple_rule_pairs
    )
)

print(
    "Pairs with repeated evidence from only one rule:",
    len(
        repeated_same_rule_pairs
    )
)

print(
    "Maximum evidence rows for one pair:",
    pair_evidence_summary[
        "evidence_count"
    ].max()
)

print(
    "Maximum distinct rules for one pair:",
    pair_evidence_summary[
        "distinct_evidence_rules"
    ].max()
)

Total distinct entity pairs: 315
Pairs with multiple evidence rows: 122
Pairs supported by multiple distinct rules: 114
Pairs with repeated evidence from only one rule: 8
Maximum evidence rows for one pair: 10
Maximum distinct rules for one pair: 5


In [39]:
# ==========================================================
# RULE-COMBINATION FREQUENCIES
# ==========================================================

rule_combination_counts = (
    pair_evidence_summary
    .groupby(
        [
            "rule_combination",
            "distinct_evidence_rules"
        ],
        dropna=False
    )
    .agg(
        pair_count=(
            "entity_id_a",
            "count"
        ),

        total_evidence_rows=(
            "evidence_count",
            "sum"
        ),

        min_evidence_rows=(
            "evidence_count",
            "min"
        ),

        max_evidence_rows=(
            "evidence_count",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        [
            "distinct_evidence_rules",
            "pair_count"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

rule_combination_counts

,rule_combination,distinct_evidence_rules,pair_count,total_evidence_rows,min_evidence_rows,max_evidence_rows
0,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_TAXPAYE...,5,2,13,5,8
1,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_OWNER_P...,5,1,5,5,5
2,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_OWNER_P...,4,2,8,4,4
3,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_OWNER_P...,4,1,4,4,4
4,SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER + SAME_P...,4,1,4,4,4
5,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_TAXPAYE...,3,6,18,3,3
6,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_OWNER_P...,3,4,12,3,3
7,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_TAX_SEN...,3,3,9,3,3
8,SAME_PARCEL_CO_OWNER + SHARED_OWNER_ADDRESS + ...,3,3,14,3,7
9,SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER + SAME_P...,3,2,6,3,3


In [40]:
multi_rule_combination_counts = (
    rule_combination_counts[
        rule_combination_counts[
            "distinct_evidence_rules"
        ] > 1
    ]
    .reset_index(drop=True)
)

print(
    "Distinct multi-rule combinations:",
    len(
        multi_rule_combination_counts
    )
)

multi_rule_combination_counts

Distinct multi-rule combinations: 17


,rule_combination,distinct_evidence_rules,pair_count,total_evidence_rows,min_evidence_rows,max_evidence_rows
0,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_TAXPAYE...,5,2,13,5,8
1,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_OWNER_P...,5,1,5,5,5
2,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_OWNER_P...,4,2,8,4,4
3,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_OWNER_P...,4,1,4,4,4
4,SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER + SAME_P...,4,1,4,4,4
5,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_TAXPAYE...,3,6,18,3,3
6,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_OWNER_P...,3,4,12,3,3
7,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_TAX_SEN...,3,3,9,3,3
8,SAME_PARCEL_CO_OWNER + SHARED_OWNER_ADDRESS + ...,3,3,14,3,7
9,SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER + SAME_P...,3,2,6,3,3


In [41]:
# ==========================================================
# STRENGTH-COMBINATION AUDIT
# ==========================================================

strength_combination_counts = (
    pair_evidence_summary
    .assign(
        strength_combination=
            pair_evidence_summary[
                "evidence_strengths"
            ]
            .apply(
                lambda strengths:
                    " + ".join(
                        strengths
                    )
            )
    )
    .groupby(
        [
            "strongest_evidence_strength",
            "strength_combination"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="pair_count"
    )
    .sort_values(
        [
            "strongest_evidence_strength",
            "pair_count"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

strength_combination_counts

,strongest_evidence_strength,strength_combination,pair_count
0,High,High + Medium,84
1,High,High,65
2,High,High + Medium + Low,2
3,Low,Low,37
4,Medium,Medium,127


In [42]:
# ==========================================================
# MULTI-RULE PAIR SUMMARY
# ==========================================================

multi_rule_pair_summary = (
    multiple_rule_pairs[
        [
            "entity_id_a",
            "entity_name_a",
            "entity_type_a",
            "entity_id_b",
            "entity_name_b",
            "entity_type_b",
            "evidence_count",
            "distinct_evidence_rules",
            "evidence_rules",
            "evidence_strengths",
            "strongest_evidence_strength",
            "rule_combination"
        ]
    ]
    .sort_values(
        [
            "distinct_evidence_rules",
            "evidence_count",
            "entity_name_a",
            "entity_name_b"
        ],
        ascending=[
            False,
            False,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

multi_rule_pair_summary

,entity_id_a,entity_name_a,entity_type_a,entity_id_b,entity_name_b,entity_type_b,evidence_count,distinct_evidence_rules,evidence_rules,evidence_strengths,strongest_evidence_strength,rule_combination
0,1345,DORMOUSE THEATRE LLC,Organization,4316,STEVE DUPUIE,Person,8,5,"(SAME_PARCEL_CO_OWNER, SAME_PARCEL_ORG_TAXPAYE...","(High, Medium)",High,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_TAXPAYE...
1,81,ABSOLUTE HEAVEN LLC,Organization,2354,"KAVANAUGH, JAMES J",Person,5,5,"(SAME_PARCEL_CO_OWNER, SAME_PARCEL_ORG_OWNER_P...","(High, Medium)",High,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_OWNER_P...
2,2532,"LADERE, SCOTT E",Person,3940,"S.T.D. MANAGEMENT, LLC",Organization,5,5,"(SAME_PARCEL_CO_OWNER, SAME_PARCEL_ORG_TAXPAYE...","(High, Medium)",High,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_TAXPAYE...
3,1254,"DECKER, LYDIA",Person,1256,"DECKER, LYDIA LIVING TRUST",Organization,4,4,"(SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER, SAME_P...","(High, Medium)",High,SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER + SAME_P...
4,2161,HUNTINGTON MORTGAGE,Organization,2398,"KERNELL, BRIANT",Person,4,4,"(SAME_PARCEL_CO_OWNER, SAME_PARCEL_ORG_OWNER_P...","(High, Medium)",High,SAME_PARCEL_CO_OWNER + SAME_PARCEL_ORG_OWNER_P...
...,...,...,...,...,...,...,...,...,...,...,...,...
109,4161,"SMALL, LYLE",Person,4162,"SMALL, LYLE ESTATE OF",Organization,2,2,"(SHARED_OWNER_ADDRESS, SHARED_TAXPAYER_ADDRESS)","(Medium,)",Medium,SHARED_OWNER_ADDRESS + SHARED_TAXPAYER_ADDRESS
110,4458,THE WALSH DRUMM FAMILY TRUST,Organization,4763,"WALSH, STEPHEN J",Person,2,2,"(SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER, SAME_P...","(High, Medium)",High,SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER + SAME_P...
111,4736,"WAGENFELD, DAVID",Person,4737,"WAGENFELD, JEANNE K CO-TRUSTEE",Organization,2,2,"(SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER, SAME_P...","(High, Medium)",High,SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER + SAME_P...
112,4736,"WAGENFELD, DAVID",Person,4738,"WAGENFELD, JEANNE K REV TRUST",Organization,2,2,"(SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER, SAME_P...","(High, Medium)",High,SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER + SAME_P...


In [43]:
# ==========================================================
# DETAILED EVIDENCE FOR MULTI-RULE PAIRS
# ==========================================================

multi_rule_pair_keys = (
    multiple_rule_pairs[
        [
            "entity_id_a",
            "entity_id_b"
        ]
    ]
)

multi_rule_evidence_detail = (
    association_evidence
    .merge(
        multi_rule_pair_keys,
        on=[
            "entity_id_a",
            "entity_id_b"
        ],
        how="inner",
        validate="many_to_one"
    )
    .merge(
        entity_a_lookup,
        on="entity_id_a",
        how="left",
        validate="many_to_one"
    )
    .merge(
        entity_b_lookup,
        on="entity_id_b",
        how="left",
        validate="many_to_one"
    )
    .sort_values(
        [
            "entity_id_a",
            "entity_id_b",
            "evidence_strength",
            "evidence_rule",
            "evidence_id"
        ]
    )
    .reset_index(drop=True)
)

multi_rule_evidence_detail

,evidence_id,entity_id_a,entity_id_b,evidence_rule,evidence_strength,parcel_number_a,parcel_number_b,role_a,role_b,shared_address,address_type,entity_name_a,entity_type_a,entity_name_b,entity_type_b
0,3,1,5062,SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER,High,06-14-431-010,06-14-431-010,Owner,Taxpayer,None,None,"""E"" STREET, INC.",Organization,"YOUNG, CAVEL J",Person
1,4,1,5062,SAME_PARCEL_ORG_OWNER_PERSON_TAX_SEND_TO,Medium,06-14-431-010,06-14-431-010,Owner,Tax Send To,None,None,"""E"" STREET, INC.",Organization,"YOUNG, CAVEL J",Person
2,13,27,5011,SAME_PARCEL_CO_OWNER,High,14-18-454-020,14-18-454-020,Owner,Owner,None,None,232 GRAND LLC,Organization,"WOODHAMS, HOLLY",Person
3,14,27,5011,SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER,Medium,14-18-454-020,14-18-454-020,Tax Send To,Owner,None,None,232 GRAND LLC,Organization,"WOODHAMS, HOLLY",Person
4,15,27,5012,SAME_PARCEL_CO_OWNER,High,14-18-454-020,14-18-454-020,Owner,Owner,None,None,232 GRAND LLC,Organization,"WOODHAMS, TIM",Person
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276,320,4801,4802,SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER,High,06-15-120-002,06-15-120-002,Taxpayer,Owner,None,None,WEATHERS LOVE REVOCABLE TRUST,Organization,"WEATHERS, SHIRLEY",Person
277,321,4801,4802,SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER,Medium,06-15-120-002,06-15-120-002,Tax Send To,Owner,None,None,WEATHERS LOVE REVOCABLE TRUST,Organization,"WEATHERS, SHIRLEY",Person
278,322,4867,4915,SAME_PARCEL_CO_OWNER,High,06-10-318-063,06-10-318-063,Owner,Owner,None,None,WHEN U NEED IT HANDYMAN LLC,Organization,"WILLIAMS, ALBERT D",Person
279,323,4867,4915,SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER,High,06-10-318-063,06-10-318-063,Owner,Taxpayer,None,None,WHEN U NEED IT HANDYMAN LLC,Organization,"WILLIAMS, ALBERT D",Person


In [44]:
# ==========================================================
# EVIDENCE-COUNT DISTRIBUTION
# ==========================================================

evidence_count_distribution = (
    pair_evidence_summary[
        "evidence_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "evidence_rows_per_pair"
    )
    .reset_index(
        name="pair_count"
    )
)

evidence_count_distribution

,evidence_rows_per_pair,pair_count
0,1,193
1,2,94
2,3,17
3,4,6
4,5,2
5,7,1
6,8,1
7,10,1


In [45]:
sum(entity_associations["evidence_count"]) == 491

True

In [46]:
# ==========================================================
# POST-CLEANUP STAGE 6 AUDIT SANITY CHECKS
# ==========================================================

assert len(
    pair_evidence_summary
) == 315

assert (
    pair_evidence_summary[
        "evidence_count"
    ].sum()
    == 491
)

assert (
    pair_evidence_summary[
        "strongest_evidence_strength"
    ]
    .isin(
        [
            "High",
            "Medium",
            "Low"
        ]
    )
).all()

assert not pair_evidence_summary.duplicated(
    subset=[
        "entity_id_a",
        "entity_id_b"
    ]
).any()

print(
    "Stage 6 pre-implementation audit "
    "sanity checks passed."
)

Stage 6 pre-implementation audit sanity checks passed.


## Stage 6 entity association validation

In [47]:
print(
    "Entity association rows:",
    len(entity_associations)
)

print(
    "\nAssociation confidence:"
)

print(
    entity_associations[
        "association_confidence"
    ].value_counts()
)

print(
    "\nTotal underlying evidence rows:",
    entity_associations[
        "evidence_count"
    ].sum()
)

print(
    "Maximum evidence rows for one association:",
    entity_associations[
        "evidence_count"
    ].max()
)

print(
    "Maximum distinct evidence rules:",
    entity_associations[
        "evidence_rule_count"
    ].max()
)


# ----------------------------------------------------------
# Basic Stage 6 grain / ID validation
# ----------------------------------------------------------

assert len(
    entity_associations
) == 315

assert entity_associations[
    "association_id"
].is_unique

assert not entity_associations.duplicated(
    subset=[
        "entity_id_a",
        "entity_id_b"
    ]
).any()

assert (
    entity_associations[
        "entity_id_a"
    ]
    <
    entity_associations[
        "entity_id_b"
    ]
).all()


# ----------------------------------------------------------
# Every referenced entity must exist
# ----------------------------------------------------------

valid_entity_ids = set(
    entities[
        "entity_id"
    ]
)

assert set(
    entity_associations[
        "entity_id_a"
    ]
).issubset(
    valid_entity_ids
)

assert set(
    entity_associations[
        "entity_id_b"
    ]
).issubset(
    valid_entity_ids
)


# ----------------------------------------------------------
# Evidence counts must reconcile exactly to Stage 5
# ----------------------------------------------------------

assert (
    entity_associations[
        "evidence_count"
    ].sum()
    ==
    len(
        association_evidence
    )
)

assert (
    entity_associations[
        "evidence_count"
    ].sum()
    == 491
)

assert (
    entity_associations[
        "evidence_count"
    ]
    >= 1
).all()

assert (
    entity_associations[
        "evidence_rule_count"
    ]
    >= 1
).all()

assert (
    entity_associations[
        "evidence_rule_count"
    ]
    <=
    entity_associations[
        "evidence_count"
    ]
).all()


# ----------------------------------------------------------
# Expected confidence distribution
# ----------------------------------------------------------

confidence_counts = (
    entity_associations[
        "association_confidence"
    ]
    .value_counts()
    .to_dict()
)

assert confidence_counts == {
    "High": 151,
    "Medium": 127,
    "Low": 37
}


# ----------------------------------------------------------
# Recalculate strongest evidence independently and confirm
# Stage 6 confidence matches it exactly.
# ----------------------------------------------------------

strength_rank = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

rank_strength = {
    1: "Low",
    2: "Medium",
    3: "High"
}

expected_confidence = (
    association_evidence
    .assign(
        _strength_rank=
            association_evidence[
                "evidence_strength"
            ]
            .map(
                strength_rank
            )
    )
    .groupby(
        [
            "entity_id_a",
            "entity_id_b"
        ],
        as_index=False
    )
    .agg(
        expected_strength_rank=(
            "_strength_rank",
            "max"
        )
    )
)

expected_confidence[
    "expected_confidence"
] = (
    expected_confidence[
        "expected_strength_rank"
    ]
    .map(
        rank_strength
    )
)

confidence_check = (
    entity_associations[
        [
            "entity_id_a",
            "entity_id_b",
            "association_confidence"
        ]
    ]
    .merge(
        expected_confidence[
            [
                "entity_id_a",
                "entity_id_b",
                "expected_confidence"
            ]
        ],
        on=[
            "entity_id_a",
            "entity_id_b"
        ],
        how="outer",
        validate="one_to_one",
        indicator=True
    )
)

assert (
    confidence_check[
        "_merge"
    ]
    == "both"
).all()

assert (
    confidence_check[
        "association_confidence"
    ]
    ==
    confidence_check[
        "expected_confidence"
    ]
).all()


# ----------------------------------------------------------
# Independently validate evidence_count and rule_count
# ----------------------------------------------------------

expected_counts = (
    association_evidence
    .groupby(
        [
            "entity_id_a",
            "entity_id_b"
        ],
        as_index=False
    )
    .agg(
        expected_evidence_count=(
            "evidence_id",
            "count"
        ),

        expected_rule_count=(
            "evidence_rule",
            "nunique"
        )
    )
)

count_check = (
    entity_associations[
        [
            "entity_id_a",
            "entity_id_b",
            "evidence_count",
            "evidence_rule_count"
        ]
    ]
    .merge(
        expected_counts,
        on=[
            "entity_id_a",
            "entity_id_b"
        ],
        how="outer",
        validate="one_to_one",
        indicator=True
    )
)

assert (
    count_check[
        "_merge"
    ]
    == "both"
).all()

assert (
    count_check[
        "evidence_count"
    ]
    ==
    count_check[
        "expected_evidence_count"
    ]
).all()

assert (
    count_check[
        "evidence_rule_count"
    ]
    ==
    count_check[
        "expected_rule_count"
    ]
).all()


print(
    "\nStage 6 entity-association validation passed."
)

Entity association rows: 315

Association confidence:
association_confidence
High      151
Medium    127
Low        37
Name: count, dtype: int64

Total underlying evidence rows: 491
Maximum evidence rows for one association: 10
Maximum distinct evidence rules: 5

Stage 6 entity-association validation passed.


## Final database tables audit

In [168]:
print("CLEAN_DF")
print(clean_df.dtypes)
print(clean_df.columns.tolist())

print("\nENTITIES")
print(entities.dtypes)
print(entities.columns.tolist())

print("\nPARCEL ENTITY ROLES")
print(parcel_entity_roles.dtypes)
print(parcel_entity_roles.columns.tolist())

print("\nASSOCIATION EVIDENCE")
print(association_evidence.dtypes)
print(association_evidence.columns.tolist())

print("\nENTITY ASSOCIATIONS")
print(entity_associations.dtypes)
print(entity_associations.columns.tolist())

print("\nENTITY OCCURRENCES")
print(entity_occurrences.dtypes)
print(entity_occurrences.columns.tolist())

CLEAN_DF
parcel_number                                 str
property_address                              str
owner_name                                    str
owner_name_2                                  str
owner_care_of                                 str
owner_address                                 str
taxpayer_name                                 str
taxpayer_care_of                              str
taxpayer_address                              str
tax_send_to_name                              str
principal_residence_exemption_percentage    int64
property_class                                str
tax_value                                   int64
assessed_value                              int64
state_equalized_value                       int64
dtype: object
['parcel_number', 'property_address', 'owner_name', 'owner_name_2', 'owner_care_of', 'owner_address', 'taxpayer_name', 'taxpayer_care_of', 'taxpayer_address', 'tax_send_to_name', 'principal_residence_exemption_percentage', 'pro

In [169]:
# ==========================================================
# DATABASE SCHEMA RANGE AUDIT
# ==========================================================

def max_string_length(df, column):
    return (
        df[column]
        .dropna()
        .astype(str)
        .str.len()
        .max()
    )


print("PARCELS")
print(
    "parcel_number max length:",
    max_string_length(
        clean_df,
        "parcel_number"
    )
)

print(
    "property_address max length:",
    max_string_length(
        clean_df,
        "property_address"
    )
)

print(
    "property_class max length:",
    max_string_length(
        clean_df,
        "property_class"
    )
)

print(
    "PRE range:",
    clean_df[
        "principal_residence_exemption_percentage"
    ].min(),
    "to",
    clean_df[
        "principal_residence_exemption_percentage"
    ].max()
)

print(
    "tax_value range:",
    clean_df["tax_value"].min(),
    "to",
    clean_df["tax_value"].max()
)

print(
    "assessed_value range:",
    clean_df["assessed_value"].min(),
    "to",
    clean_df["assessed_value"].max()
)

print(
    "state_equalized_value range:",
    clean_df[
        "state_equalized_value"
    ].min(),
    "to",
    clean_df[
        "state_equalized_value"
    ].max()
)


print("\nENTITIES")

for column in [
    "canonical_name",
    "canonical_address",
    "entity_type",
    "organization_type",
    "organization_classification_confidence"
]:
    print(
        f"{column} max length:",
        max_string_length(
            entities,
            column
        )
    )


print("\nASSOCIATION EVIDENCE")

for column in [
    "evidence_rule",
    "evidence_strength",
    "role_a",
    "role_b",
    "shared_address",
    "address_type"
]:
    print(
        f"{column} max length:",
        max_string_length(
            association_evidence,
            column
        )
    )


print("\nENTITY OCCURRENCES")

for column in [
    "entity_name",
    "associated_address",
    "role",
    "source_entity_name",
    "source_column",
    "owner_line_continuation_rule",
    "entity_type",
    "joint_parse_rule",
    "joint_parse_confidence",
    "organization_type",
    "organization_classification_confidence",
    "address_source"
]:
    print(
        f"{column} max length:",
        max_string_length(
            entity_occurrences,
            column
        )
    )

print(
    "joint_party_index max:",
    entity_occurrences[
        "joint_party_index"
    ].max()
)

print(
    "joint_party_count max:",
    entity_occurrences[
        "joint_party_count"
    ].max()
)

PARCELS
parcel_number max length: 13
property_address max length: 47
property_class max length: 30
PRE range: 0 to 100
tax_value range: 0 to 6055777
assessed_value range: 0 to 7729000
state_equalized_value range: 0 to 7729000

ENTITIES
canonical_name max length: 50
canonical_address max length: 63
entity_type max length: 12
organization_type max length: 24
organization_classification_confidence max length: 6

ASSOCIATION EVIDENCE
evidence_rule max length: 42
evidence_strength max length: 6
role_a max length: 13
role_b max length: 13
shared_address max length: 49
address_type max length: 8

ENTITY OCCURRENCES
entity_name max length: 50
associated_address max length: 63
role max length: 16
source_entity_name max length: 37
source_column max length: 16
owner_line_continuation_rule max length: 31
entity_type max length: 12
joint_parse_rule max length: 35
joint_parse_confidence max length: 6
organization_type max length: 24
organization_classification_confidence max length: 6
address_source

## Owner address audit

In [49]:
owner_address_entities = (
    entity_occurrences[
        (
            entity_occurrences["role"]
            == "Owner"
        )
        &
        (
            entity_occurrences[
                "associated_address"
            ].notna()
        )
        &
        (
            entity_occurrences[
                "associated_address"
            ].str.strip()
            != ""
        )
    ]
    [
        [
            "parcel_number",
            "entity_id",
            "associated_address"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "associated_address":
                "owner_address"
        }
    )
)

In [50]:
owner_address_entities = (
    owner_address_entities
    .merge(
        entities[
            [
                "entity_id",
                "entity_type",
                "canonical_name"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one"
    )
)

In [51]:
owner_address_profile = (
    owner_address_entities
    .groupby(
        "owner_address"
    )
    .agg(
        distinct_entities=(
            "entity_id",
            "nunique"
        ),
        distinct_parcels=(
            "parcel_number",
            "nunique"
        ),
        distinct_person_entities=(
            "entity_id",
            lambda ids:
                owner_address_entities.loc[
                    ids.index
                ]
                .loc[
                    lambda df:
                        df["entity_type"]
                        == "Person",
                    "entity_id"
                ]
                .nunique()
        ),
        distinct_organization_entities=(
            "entity_id",
            lambda ids:
                owner_address_entities.loc[
                    ids.index
                ]
                .loc[
                    lambda df:
                        df["entity_type"]
                        == "Organization",
                    "entity_id"
                ]
                .nunique()
        )
    )
    .reset_index()
)

owner_address_profile.head(30)

,owner_address,distinct_entities,distinct_parcels,distinct_person_entities,distinct_organization_entities
0,"07475 52 1/2 ST, GRAND JUNCTION, MI 49056",1,1,0,1
1,"07791 56TH ST, GRAND JUNCTION, MI 49056",3,5,3,0
2,"1 EQUITY WAY, WESTLAKE, OH 44145",1,1,1,0
3,"1 HOME CAMPUS, DES MOINES, IA 50328",1,1,0,1
4,"100 E BATTLE CREEK ST, GALESBURG, MI 49053",1,1,1,0
5,"100 E KILGORE RD, PORTAGE, MI 49002-5584",1,1,0,1
6,"100 E PINE ST STE 110, ORLANDO, FL 32801",1,1,0,1
7,"100 KIMBALL PL STE 600, ALPHARETTA, GA 30009-2599",1,1,0,1
8,"100 MONROE ST, KALAMAZOO, MI 49006",1,5,0,1
9,"100 W MICHIGAN AVE STE 200, KALAMAZOO, MI 49007",1,1,0,1


In [52]:
owner_address_profile[
    [
        "distinct_entities",
        "distinct_parcels"
    ]
].describe()

,distinct_entities,distinct_parcels
count,3144.000000,3144.000000
mean,1.330153,1.232188
std,0.572268,0.953668
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,1.000000
max,7.000000,21.000000


In [53]:
owner_address_persons = (
    owner_address_entities[
        owner_address_entities[
            "entity_type"
        ] == "Person"
    ]
    .rename(
        columns={
            "entity_id":
                "person_id",
            "canonical_name":
                "person_name",
            "parcel_number":
                "person_parcel"
        }
    )
)

owner_address_organizations = (
    owner_address_entities[
        owner_address_entities[
            "entity_type"
        ] == "Organization"
    ]
    .rename(
        columns={
            "entity_id":
                "organization_id",
            "canonical_name":
                "organization_name",
            "parcel_number":
                "organization_parcel"
        }
    )
)

shared_owner_address_candidates = (
    owner_address_organizations
    .merge(
        owner_address_persons,
        on="owner_address",
        how="inner"
    )
)

In [54]:
shared_owner_address_candidates = (
    shared_owner_address_candidates[
        shared_owner_address_candidates[
            "organization_parcel"
        ]
        !=
        shared_owner_address_candidates[
            "person_parcel"
        ]
    ]
    .copy()
)

In [55]:
print(
    "Cross-parcel shared-owner-address evidence rows:",
    len(shared_owner_address_candidates)
)

print(
    "Distinct Person-Organization pairs:",
    shared_owner_address_candidates[
        [
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct shared owner addresses:",
    shared_owner_address_candidates[
        "owner_address"
    ].nunique()
)

Cross-parcel shared-owner-address evidence rows: 246
Distinct Person-Organization pairs: 97
Distinct shared owner addresses: 52


In [56]:
candidate_address_degree = (
    shared_owner_address_candidates[
        [
            "owner_address"
        ]
    ]
    .drop_duplicates()
    .merge(
        owner_address_profile,
        on="owner_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels"
        ],
        ascending=False
    )
)

candidate_address_degree.head(30)

,owner_address,distinct_entities,distinct_parcels,distinct_person_entities,distinct_organization_entities
40,"12265 E S AVE, SCOTTS, MI 49088",6,5,4,2
44,"8713 S 38TH ST, SCOTTS, MI 49088",5,18,2,1
38,"PO BOX 366, GALESBURG, MI 49053",5,8,3,2
36,"11215 E G AVE, GALESBURG, MI 49053",5,3,2,3
5,"1417 ACADEMY, KALAMAZOO, MI 49006",4,6,1,3
45,"1627 CLYDE ST, VICKSBURG, MI 49097",4,3,3,1
9,"1420 N ROSE ST, KALAMAZOO, MI 49007",4,2,2,2
15,"6335 E H AVE, KALAMAZOO, MI 49048",4,2,3,1
12,"1362 FERNDALE AVE, KALAMAZOO, MI 49006",3,7,2,1
46,"15892 S 24TH ST, VICKSBURG, MI 49097",3,6,2,1


In [57]:
owner_address_candidate_detail = (
    shared_owner_address_candidates[
        [
            "owner_address",
            "person_id",
            "person_name",
            "person_parcel",
            "organization_id",
            "organization_name",
            "organization_parcel"
        ]
    ]
    .merge(
        owner_address_profile[
            [
                "owner_address",
                "distinct_entities",
                "distinct_parcels"
            ]
        ],
        on="owner_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels",
            "owner_address",
            "organization_name",
            "person_name"
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

owner_address_candidate_detail

,owner_address,person_id,person_name,person_parcel,organization_id,organization_name,organization_parcel,distinct_entities,distinct_parcels
0,"12265 E S AVE, SCOTTS, MI 49088",3501,"PERRIN, BONNA M",12-29-380-012,3497,PERRIN ROBB & DANA TRUST,08-23-476-025,6,5
1,"12265 E S AVE, SCOTTS, MI 49088",3503,"PERRIN, DANA L",08-23-476-021,3497,PERRIN ROBB & DANA TRUST,08-23-476-025,6,5
2,"12265 E S AVE, SCOTTS, MI 49088",3506,"PERRIN, JERROLD F",12-29-380-012,3497,PERRIN ROBB & DANA TRUST,08-23-476-025,6,5
3,"12265 E S AVE, SCOTTS, MI 49088",3508,"PERRIN, ROBB J",08-23-476-021,3497,PERRIN ROBB & DANA TRUST,08-23-476-025,6,5
4,"12265 E S AVE, SCOTTS, MI 49088",3508,"PERRIN, ROBB J",12-29-355-011,3497,PERRIN ROBB & DANA TRUST,08-23-476-025,6,5
...,...,...,...,...,...,...,...,...,...
241,"3606 N WESTNEDGE AVE, KALAMAZOO, MI 49004",699,"BRUBAKER, COLLEEN",06-03-105-031,700,"BRUBAKER, MICHAEL DALE ESTATE",06-03-105-032,2,2
242,"3704 OLNEY RD, KALAMAZOO, MI 49006",518,"BOLDEN, BRITNEY N.",06-18-265-160,5058,"YOUNG & YOUNG, L.L.C.",06-10-309-007,2,2
243,"4205 AMELIA WAY, NAPLES, FL 34119-9043",4143,"SKARTSIARIS, THEODORE E, TRUSTEE",06-15-354-130,4144,"SKARTSIARIS, THEODORE MARITAL TRST",06-34-365-001,2,2
244,"690 N 1ST ST, KALAMAZOO, MI 49009",963,CHERYL RAVEN-CRAWFORD,05-18-405-041,965,CHIPMAN ANN D TRUST,05-18-405-011,2,2


In [58]:
person_address_members = (
    owner_address_persons[
        [
            "owner_address",
            "person_id",
            "person_name",
            "person_parcel"
        ]
    ]
    .rename(
        columns={
            "person_id": "entity_id",
            "person_name": "entity_name",
            "person_parcel": "parcel_number"
        }
    )
    .assign(
        entity_type="Person"
    )
)

organization_address_members = (
    owner_address_organizations[
        [
            "owner_address",
            "organization_id",
            "organization_name",
            "organization_parcel"
        ]
    ]
    .rename(
        columns={
            "organization_id": "entity_id",
            "organization_name": "entity_name",
            "organization_parcel": "parcel_number"
        }
    )
    .assign(
        entity_type="Organization"
    )
)

candidate_addresses = set(
    shared_owner_address_candidates[
        "owner_address"
    ]
)

owner_address_members = (
    pd.concat(
        [
            person_address_members,
            organization_address_members
        ],
        ignore_index=True
    )
)

owner_address_members = (
    owner_address_members[
        owner_address_members[
            "owner_address"
        ].isin(candidate_addresses)
    ]
    .drop_duplicates()
)

In [59]:
owner_address_member_summary = (
    owner_address_members
    .groupby(
        [
            "owner_address",
            "entity_id",
            "entity_name",
            "entity_type"
        ],
        dropna=False
    )
    .agg(
        parcel_count=(
            "parcel_number",
            "nunique"
        ),
        parcels=(
            "parcel_number",
            lambda x:
                ", ".join(
                    sorted(set(x))
                )
        )
    )
    .reset_index()
    .merge(
        owner_address_profile[
            [
                "owner_address",
                "distinct_entities",
                "distinct_parcels"
            ]
        ],
        on="owner_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels",
            "owner_address",
            "entity_type",
            "entity_name"
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

owner_address_member_summary

,owner_address,entity_id,entity_name,entity_type,parcel_count,parcels,distinct_entities,distinct_parcels
0,"12265 E S AVE, SCOTTS, MI 49088",3497,PERRIN ROBB & DANA TRUST,Organization,1,08-23-476-025,6,5
1,"12265 E S AVE, SCOTTS, MI 49088",3498,PERRIN ROBB J & DANA L TRUST,Organization,1,08-23-476-018,6,5
2,"12265 E S AVE, SCOTTS, MI 49088",3501,"PERRIN, BONNA M",Person,1,12-29-380-012,6,5
3,"12265 E S AVE, SCOTTS, MI 49088",3503,"PERRIN, DANA L",Person,1,08-23-476-021,6,5
4,"12265 E S AVE, SCOTTS, MI 49088",3506,"PERRIN, JERROLD F",Person,1,12-29-380-012,6,5
...,...,...,...,...,...,...,...,...
139,"4205 AMELIA WAY, NAPLES, FL 34119-9043",4143,"SKARTSIARIS, THEODORE E, TRUSTEE",Person,1,06-15-354-130,2,2
140,"690 N 1ST ST, KALAMAZOO, MI 49009",965,CHIPMAN ANN D TRUST,Organization,1,05-18-405-011,2,2
141,"690 N 1ST ST, KALAMAZOO, MI 49009",963,CHERYL RAVEN-CRAWFORD,Person,1,05-18-405-041,2,2
142,"939 FIELD AVE, KALAMAZOO, MI 49048",5037,"WYMAN, PATRICK ESTATE",Organization,1,06-24-305-951,2,2


## Audit of role combinations between different entities on the same parcel

In [60]:
parcel_role_pairs = (
    parcel_entity_roles
    .merge(
        parcel_entity_roles,
        on="parcel_number",
        suffixes=("_a", "_b")
    )
)

# Remove self-pairs and keep each unordered entity pair once
parcel_role_pairs = (
    parcel_role_pairs[
        parcel_role_pairs["entity_id_a"]
        < parcel_role_pairs["entity_id_b"]
    ]
    .copy()
)

In [61]:
entity_lookup_a = (
    entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type",
            "organization_type"
        ]
    ]
    .rename(
        columns={
            "entity_id": "entity_id_a",
            "canonical_name": "name_a",
            "entity_type": "entity_type_a",
            "organization_type": "organization_type_a"
        }
    )
)

entity_lookup_b = (
    entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type",
            "organization_type"
        ]
    ]
    .rename(
        columns={
            "entity_id": "entity_id_b",
            "canonical_name": "name_b",
            "entity_type": "entity_type_b",
            "organization_type": "organization_type_b"
        }
    )
)

parcel_role_pairs = (
    parcel_role_pairs
    .merge(
        entity_lookup_a,
        on="entity_id_a",
        how="left"
    )
    .merge(
        entity_lookup_b,
        on="entity_id_b",
        how="left"
    )
)

In [62]:
parcel_role_pairs["role_pair"] = (
    parcel_role_pairs[
        ["role_a", "role_b"]
    ]
    .apply(
        lambda row: " + ".join(
            sorted(row)
        ),
        axis=1
    )
)

In [63]:
parcel_role_pairs["entity_type_pair"] = (
    parcel_role_pairs[
        [
            "entity_type_a",
            "entity_type_b"
        ]
    ]
    .apply(
        lambda row: " + ".join(
            sorted(row)
        ),
        axis=1
    )
)

In [64]:
role_pair_counts = (
    parcel_role_pairs[
        "role_pair"
    ]
    .value_counts()
    .rename_axis("role_pair")
    .reset_index(name="count")
)

role_pair_counts

,role_pair,count
0,Owner + Tax Send To,2465
1,Owner + Taxpayer,1303
2,Owner + Owner,1034
3,Tax Send To + Tax Send To,727
4,Tax Send To + Taxpayer,690
5,Taxpayer + Taxpayer,262
6,Owner + Owner Care Of,249
7,Owner Care Of + Tax Send To,224
8,Owner + Taxpayer Care Of,186
9,Owner Care Of + Taxpayer,185


In [65]:
role_type_pair_counts = (
    parcel_role_pairs
    .groupby(
        [
            "role_pair",
            "entity_type_pair"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
)

role_type_pair_counts

,role_pair,entity_type_pair,count
23,Owner + Tax Send To,Person + Person,2101
31,Owner + Taxpayer,Person + Person,1015
7,Owner + Owner,Person + Person,914
60,Tax Send To + Tax Send To,Person + Person,727
66,Tax Send To + Taxpayer,Person + Person,606
...,...,...,...
33,Owner + Taxpayer Care Of,Joint + Joint,1
62,Tax Send To + Taxpayer,Joint + Organization,1
53,Owner Care Of + Taxpayer,Organization + Unknown,1
55,Owner Care Of + Taxpayer,Person + Unknown,1


In [66]:
print(
    "Parcels containing more than one entity:",
    parcel_role_pairs[
        "parcel_number"
    ].nunique()
)

print(
    "Distinct same-parcel entity pairs:",
    parcel_role_pairs[
        [
            "parcel_number",
            "entity_id_a",
            "entity_id_b"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

Parcels containing more than one entity: 1561
Distinct same-parcel entity pairs: 3042


In [67]:
org_person_pairs = (
    parcel_role_pairs[
        (
            (
                parcel_role_pairs["entity_type_a"]
                == "Organization"
            )
            &
            (
                parcel_role_pairs["entity_type_b"]
                == "Person"
            )
        )
        |
        (
            (
                parcel_role_pairs["entity_type_a"]
                == "Person"
            )
            &
            (
                parcel_role_pairs["entity_type_b"]
                == "Organization"
            )
        )
    ]
    .copy()
)

In [68]:
org_person_pairs["organization_id"] = (
    org_person_pairs.apply(
        lambda row:
            row["entity_id_a"]
            if row["entity_type_a"] == "Organization"
            else row["entity_id_b"],
        axis=1
    )
)

org_person_pairs["organization_name"] = (
    org_person_pairs.apply(
        lambda row:
            row["name_a"]
            if row["entity_type_a"] == "Organization"
            else row["name_b"],
        axis=1
    )
)

org_person_pairs["organization_role"] = (
    org_person_pairs.apply(
        lambda row:
            row["role_a"]
            if row["entity_type_a"] == "Organization"
            else row["role_b"],
        axis=1
    )
)

org_person_pairs["person_id"] = (
    org_person_pairs.apply(
        lambda row:
            row["entity_id_a"]
            if row["entity_type_a"] == "Person"
            else row["entity_id_b"],
        axis=1
    )
)

org_person_pairs["person_name"] = (
    org_person_pairs.apply(
        lambda row:
            row["name_a"]
            if row["entity_type_a"] == "Person"
            else row["name_b"],
        axis=1
    )
)

org_person_pairs["person_role"] = (
    org_person_pairs.apply(
        lambda row:
            row["role_a"]
            if row["entity_type_a"] == "Person"
            else row["role_b"],
        axis=1
    )
)

In [69]:
org_person_role_counts = (
    org_person_pairs
    .groupby(
        [
            "organization_role",
            "person_role"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
    .reset_index(drop=True)
)

org_person_role_counts

,organization_role,person_role,count
0,Owner,Taxpayer,79
1,Taxpayer,Owner,55
2,Owner,Tax Send To,49
3,Owner,Owner,49
4,Tax Send To,Owner,49
5,Owner,Owner Care Of,44
6,Tax Send To,Taxpayer,39
7,Tax Send To,Owner Care Of,37
8,Owner,Taxpayer Care Of,31
9,Tax Send To,Taxpayer Care Of,30


In [70]:
print(
    "Organization-Person evidence rows:",
    len(org_person_pairs)
)

print(
    "Distinct parcel/person/organization combinations:",
    org_person_pairs[
        [
            "parcel_number",
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct person/organization pairs across all parcels:",
    org_person_pairs[
        [
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

Organization-Person evidence rows: 593
Distinct parcel/person/organization combinations: 334
Distinct person/organization pairs across all parcels: 302


In [71]:
org_person_detail = (
    org_person_pairs[
        [
            "parcel_number",

            "person_id",
            "person_name",
            "person_role",

            "organization_id",
            "organization_name",
            "organization_type_a",
            "organization_type_b",
            "organization_role"
        ]
    ]
    .copy()
)

organization_lookup = (
    entities[
        [
            "entity_id",
            "organization_type"
        ]
    ]
    .rename(
        columns={
            "entity_id":
                "organization_id"
        }
    )
)

org_person_detail = (
    org_person_pairs[
        [
            "parcel_number",

            "person_id",
            "person_name",
            "person_role",

            "organization_id",
            "organization_name",
            "organization_role"
        ]
    ]
    .merge(
        organization_lookup,
        on="organization_id",
        how="left"
    )
    .sort_values(
        [
            "organization_role",
            "person_role",
            "parcel_number"
        ]
    )
    .reset_index(drop=True)
)

In [72]:
# Get the six most common directional role combinations
top_role_combinations = (
    org_person_role_counts
    .head(6)[
        [
            "organization_role",
            "person_role"
        ]
    ]
)

# Keep only rows belonging to those six combinations
top_role_candidates = (
    org_person_detail
    .merge(
        top_role_combinations,
        on=[
            "organization_role",
            "person_role"
        ],
        how="inner"
    )
)

# Randomize the order reproducibly, then take up to
# 10 examples from each role combination.
top_role_samples = (
    top_role_candidates
    .sample(
        frac=1,
        random_state=42
    )
    .groupby(
        [
            "organization_role",
            "person_role"
        ],
        group_keys=False,
        sort=False
    )
    .head(10)
    .sort_values(
        [
            "organization_role",
            "person_role",
            "parcel_number"
        ]
    )
    .reset_index(drop=True)
)

top_role_samples[
    [
        "parcel_number",
        "person_name",
        "person_role",
        "organization_name",
        "organization_type",
        "organization_role"
    ]
]

,parcel_number,person_name,person_role,organization_name,organization_type,organization_role
0,06-10-318-063,"WILLIAMS, ALBERT D",Owner,WHEN U NEED IT HANDYMAN LLC,LLC,Owner
1,06-13-140-010,BELENDRIA A. CUNNINGHAM,Owner,"DAVIS, STELLA P. ESTATE",Estate,Owner
2,06-14-256-003,"HILLMAN, TONY",Owner,HILLMAN LIVING TRUST,Trust,Owner
3,06-14-431-010,"BAKER, JAN E",Owner,"""E"" STREET, INC.",Corporation,Owner
4,06-22-317-010,"KAVANAUGH, JAMES J",Owner,ABSOLUTE HEAVEN LLC,LLC,Owner
5,07-24-201-480,HOSKEN WILLIAM J,Owner,THE WILLIAM J HOSKENLIV TRUST,Trust,Owner
6,07-24-215-160,"GRAHAM, WILLIAM D",Owner,THE COPPER CREEK TRUST,Trust,Owner
7,14-18-454-020,"WOODHAMS, HOLLY",Owner,232 GRAND LLC,LLC,Owner
8,14-19-209-060,"PARKER, MARK A",Owner,MARKS SALES & SERVICE LLC &,LLC,Owner
9,16-33-330-029,PRATER LEANN R,Owner,LEANN R PRATER REVOCABLE TRUST,Trust,Owner


## Investigating person-like entities

In [73]:
person_red_flag_pattern = (
    r"\b(?:"
    r"IRA|"
    r"FBO|"
    r"CUSTODIAN|"
    r"SOLE\s+PROPRIETORSHIP|"
    r"MGMT|"
    r"MANAGEMENT|"
    r"MORTGAGE|"
    r"BANK|"
    r"PROPERTIES|"
    r"HOLDINGS|"
    r"SERV|"
    r"SERVICES"
    r")\b"
)

person_classification_audit = (
    entities[
        (
            entities["entity_type"]
            == "Person"
        )
        &
        (
            entities["canonical_name"]
            .str.contains(
                person_red_flag_pattern,
                regex=True,
                na=False
            )
        )
    ]
    [
        [
            "entity_id",
            "canonical_name",
            "canonical_address",
            "occurrence_count"
        ]
    ]
    .sort_values(
        "canonical_name"
    )
    .reset_index(drop=True)
)

print(
    "Potentially suspicious Person entities:",
    len(person_classification_audit)
)

person_classification_audit

Potentially suspicious Person entities: 0


,entity_id,canonical_name,canonical_address,occurrence_count


In [74]:
suspicious_names = [
    "FBO GARY E APPS IRA",
    "FBO JOHN MEYERS IRA",
    "FBO RAMON HUERTA IRA",
    "FBO RICHARD DAUDERT IRA",
    "RUSHMORE LOAD MGT SERV"
]

entities[
    entities["canonical_name"].isin(
        suspicious_names
    )
][
    [
        "entity_id",
        "canonical_name",
        "entity_type",
        "organization_type",
        "organization_classification_confidence",
        "canonical_address",
        "occurrence_count"
    ]
].sort_values(
    [
        "canonical_name",
        "canonical_address"
    ]
)

,entity_id,canonical_name,entity_type,organization_type,organization_classification_confidence,canonical_address,occurrence_count
1507,1508,FBO GARY E APPS IRA,Organization,Retirement Account,High,"3623 KENBROOK CT, KALAMAZOO, MI 49006",1
1508,1509,FBO GARY E APPS IRA,Organization,Retirement Account,High,"813 W S ST, KALAMAZOO, MI 49007",1
1509,1510,FBO JOHN MEYERS IRA,Organization,Retirement Account,High,"471 W S ST #407, KALAMAZOO, MI 49007",1
1510,1511,FBO RAMON HUERTA IRA,Organization,Retirement Account,High,"3720 ADAMS ST, KALAMAZOO, MI 49008",1
1511,1512,FBO RICHARD DAUDERT IRA,Organization,Retirement Account,High,"4730 E CENTRE AVE, PORTAGE, MI 49002",2
3913,3914,RUSHMORE LOAD MGT SERV,Organization,Other Organization,Medium,"15480 LAGUNA CANYON RD, STE 100, IRVINE, CA 92618",1


In [75]:
entities["occurrence_count"].sum()

np.int64(12056)

## Auditing multi-comma persons

In [76]:
multi_comma_persons = (
    entities[
        (
            entities["entity_type"]
            == "Person"
        )
        &
        (
            entities["canonical_name"]
            .str.count(",")
            >= 3
        )
    ]
    [
        [
            "entity_id",
            "canonical_name",
            "canonical_address",
            "occurrence_count"
        ]
    ]
    .sort_values(
        "canonical_name"
    )
    .reset_index(drop=True)
)

print(
    "Person entities with at least 3 commas:",
    len(multi_comma_persons)
)

multi_comma_persons

Person entities with at least 3 commas: 0


,entity_id,canonical_name,canonical_address,occurrence_count


In [77]:
two_comma_persons = (
    entities[
        (
            entities["entity_type"]
            == "Person"
        )
        &
        (
            entities["canonical_name"]
            .str.count(",")
            == 2
        )
    ]
    [
        [
            "entity_id",
            "canonical_name",
            "canonical_address",
            "occurrence_count"
        ]
    ]
    .sort_values(
        "canonical_name"
    )
    .reset_index(drop=True)
)

print(
    "Person entities with exactly 2 commas:",
    len(two_comma_persons)
)

two_comma_persons

Person entities with exactly 2 commas: 4


,entity_id,canonical_name,canonical_address,occurrence_count
0,689,"BROWN, ROBERT WENDELL, JR.","1115 DUPONT AVE, KALAMAZOO, MI 49048-3213",5
1,1316,"DIXISON, MOSSY, TRUSTEE","722 MABEL ST, KALAMAZOO, MI 49007",1
2,4143,"SKARTSIARIS, THEODORE E, TRUSTEE","4205 AMELIA WAY, NAPLES, FL 34119-9043",3
3,5015,"WOODS, ETHEL GRAY, TRUSTEE","1715 COBB AVE, KALAMAZOO, MI 49007",2


## Audit parcel_entity_roles

In [78]:
print(parcel_entity_roles.head(10))

   parcel_number  entity_id         role
0  01-01-201-013       3587        Owner
1  01-01-201-013       3587  Tax Send To
2  01-01-251-013       1008        Owner
3  01-01-251-013       1008  Tax Send To
4  01-01-251-013       3841        Owner
5  01-01-251-050       1009        Owner
6  01-01-251-050       1009  Tax Send To
7  01-01-251-060       1009        Owner
8  01-01-251-060       1009  Tax Send To
9  01-01-301-050       4259        Owner


In [79]:
print(
    "Entity occurrences:",
    len(entity_occurrences)
)

print(
    "Parcel entity roles:",
    len(parcel_entity_roles)
)

print(
    "Unique parcels:",
    parcel_entity_roles[
        "parcel_number"
    ].nunique()
)

print(
    "Unique entities represented:",
    parcel_entity_roles[
        "entity_id"
    ].nunique()
)

Entity occurrences: 12056
Parcel entity roles: 12042
Unique parcels: 3874
Unique entities represented: 5078


In [80]:
duplicate_relationships = (
    parcel_entity_roles
    .duplicated(
        subset=[
            "parcel_number",
            "entity_id",
            "role"
        ]
    )
    .sum()
)

print(
    "Duplicate parcel/entity/role relationships:",
    duplicate_relationships
)

Duplicate parcel/entity/role relationships: 0


In [81]:
expected_relationships = (
    entity_occurrences[
        [
            "parcel_number",
            "entity_id",
            "role"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Expected relationships:",
    len(expected_relationships)
)

print(
    "Actual relationships:",
    len(parcel_entity_roles)
)

assert (
    len(expected_relationships)
    == len(parcel_entity_roles)
)

Expected relationships: 12042
Actual relationships: 12042


In [82]:
parcel_entity_roles[
    "role"
].value_counts()

role
Owner               4863
Tax Send To         4571
Taxpayer            2241
Owner Care Of        207
Taxpayer Care Of     160
Name: count, dtype: int64

In [83]:
parcel_entity_roles.groupby(
    "parcel_number"
)["entity_id"].nunique().describe()

count    3874.000000
mean        1.556789
std         0.838957
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         7.000000
Name: entity_id, dtype: float64

In [84]:
role_occurrence_counts = (
    entity_occurrences
    .groupby(
        [
            "parcel_number",
            "entity_id",
            "role"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="occurrence_count"
    )
)

duplicate_role_relationships = (
    role_occurrence_counts[
        role_occurrence_counts[
            "occurrence_count"
        ] > 1
    ]
    .sort_values(
        by="occurrence_count",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Parcel/entity/role relationships represented "
    "by multiple occurrences:",
    len(duplicate_role_relationships)
)

duplicate_role_relationships

Parcel/entity/role relationships represented by multiple occurrences: 14


,parcel_number,entity_id,role,occurrence_count
0,01-35-170-220,3264,Owner,2
1,06-01-205-031,2331,Owner,2
2,06-14-150-003,2926,Owner,2
3,06-16-238-084,4982,Owner,2
4,06-16-349-005,217,Owner,2
5,06-21-409-001,4763,Owner,2
6,06-23-107-013,4517,Owner,2
7,06-29-437-025,1429,Owner,2
8,06-35-147-001,1256,Owner,2
9,06-35-175-472,2538,Owner,2


In [85]:
duplicate_role_detail = (
    entity_occurrences
    .merge(
        duplicate_role_relationships[
            [
                "parcel_number",
                "entity_id",
                "role"
            ]
        ],
        on=[
            "parcel_number",
            "entity_id",
            "role"
        ],
        how="inner"
    )
    .merge(
        entities[
            [
                "entity_id",
                "canonical_name",
                "canonical_address"
            ]
        ],
        on="entity_id",
        how="left"
    )
    .sort_values(
        by=[
            "parcel_number",
            "entity_id",
            "role",
            "occurrence_id"
        ]
    )
)

duplicate_role_detail[
    [
        "parcel_number",
        "entity_id",
        "canonical_name",
        "entity_type",
        "role",
        "entity_name",
        "source_entity_name",
        "source_column",
        "associated_address",
        "joint_expanded",
        "source_occurrence_id",
        "occurrence_id"
    ]
]

,parcel_number,entity_id,canonical_name,entity_type,role,entity_name,source_entity_name,source_column,associated_address,joint_expanded,source_occurrence_id,occurrence_id
0,01-35-170-220,3264,NIEWOONDER ED & KAY TRUST,Organization,Owner,NIEWOONDER ED & KAY TRUST,NIEWOONDER ED & KAY,owner_name,"6817 COUNTRY VIEW, KALAMAZOO, MI 49009",False,57,65
14,01-35-170-220,3264,NIEWOONDER ED & KAY TRUST,Organization,Owner,NIEWOONDER ED & KAY TRUST,TRUST,owner_name_2,"6817 COUNTRY VIEW, KALAMAZOO, MI 49009",False,3878,4638
1,06-01-205-031,2331,KALAMAZOO EAST CONGREGATION OF JEHOVAH'S WITNESS,Organization,Owner,KALAMAZOO EAST CONGREGATION OF JEHOVAH'S WITNESS,KALAMAZOO EAST CONGREGATION OF,owner_name,"3414 ENTERPRISE, KALAMAZOO, MI 49004",False,485,601
15,06-01-205-031,2331,KALAMAZOO EAST CONGREGATION OF JEHOVAH'S WITNESS,Organization,Owner,KALAMAZOO EAST CONGREGATION OF JEHOVAH'S WITNESS,JEHOVAH'S WITNESS,owner_name_2,"3414 ENTERPRISE, KALAMAZOO, MI 49004",False,3894,4656
2,06-14-150-003,2926,"MCKINLEY, DEON MVP AUTO SALES",Organization,Owner,"MCKINLEY, DEON MVP AUTO SALES","MCKINLEY, DEON",owner_name,"717 RIVERVIEW DR, KALAMAZOO, MI 49048",False,1076,1293
16,06-14-150-003,2926,"MCKINLEY, DEON MVP AUTO SALES",Organization,Owner,"MCKINLEY, DEON MVP AUTO SALES",MVP AUTO SALES,owner_name_2,"717 RIVERVIEW DR, KALAMAZOO, MI 49048",False,3925,4689
3,06-16-238-084,4982,"WITHERSPOON, VIRGIL",Person,Owner,"WITHERSPOON, VIRGIL","WITHERSPOON, VIRGIL",owner_name,"535 ELIZABETH ST, KALAMAZOO, MI 49007",False,1465,1756
17,06-16-238-084,4982,"WITHERSPOON, VIRGIL",Person,Owner,"WITHERSPOON, VIRGIL","WITHERSPOON, VIRGIL",owner_name_2,"535 ELIZABETH ST, KALAMAZOO, MI 49007",False,3949,4718
4,06-16-349-005,217,"APPLEGATE, NEIL E & SHARON K REVOCABLE TRUST",Organization,Owner,"APPLEGATE, NEIL E & SHARON K REVOCABLE TRUST","APPLEGATE, NEIL E & SHARON K",owner_name,"4912 N DR S, E LEROY, MI 49051",False,1553,1856
18,06-16-349-005,217,"APPLEGATE, NEIL E & SHARON K REVOCABLE TRUST",Organization,Owner,"APPLEGATE, NEIL E & SHARON K REVOCABLE TRUST",REVOCABLE TRUST,owner_name_2,"4912 N DR S, E LEROY, MI 49051",False,3957,4726


In [86]:
print(
    "Final entities:",
    len(entities)
)

print(
    "Entities represented in parcel roles:",
    parcel_entity_roles[
        "entity_id"
    ].nunique()
)

Final entities: 5078
Entities represented in parcel roles: 5078


## Auditing merge_on_match_key

In [87]:
raw_entities_audit = (
    resolver._create_raw_entities(
        entity_occurrences
    )
)

print(
    "Raw entities:",
    len(raw_entities_audit)
)

Raw entities: 7191


In [88]:
pre_abbreviation_entities = (
    raw_entities_audit.copy()
)

pass_number = 0

while True:

    pass_number += 1

    count_before = len(
        pre_abbreviation_entities
    )

    pre_abbreviation_entities = (
        resolver._merge_blank_addresses(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_missing_street_suffix_variants(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_zip4_variants(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_equivalent_person_name_formats(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_middle_initial_variants(
            pre_abbreviation_entities
        )
    )

    count_after = len(
        pre_abbreviation_entities
    )

    print(
        f"Pass {pass_number}: "
        f"{count_before} -> {count_after}"
    )

    if count_after == count_before:
        break

Pass 1: 7191 -> 5101
Pass 2: 5101 -> 5083
Pass 3: 5083 -> 5083


In [89]:
abbreviation_audit = (
    pre_abbreviation_entities.copy()
)

abbreviation_audit[
    "match_name"
] = (
    abbreviation_audit[
        "canonical_name"
    ]
    .apply(
        resolver._expand_common_abbreviations
    )
)

In [90]:
abbreviation_candidates = []

for _, group in abbreviation_audit.groupby(
    [
        "entity_type",
        "match_name",
        "canonical_address"
    ],
    dropna=False,
    sort=False
):

    if len(group) < 2:
        continue

    original_names = tuple(
        sorted(
            set(
                group["canonical_name"]
            )
        )
    )

    # Ignore exact duplicates. We only want groups where
    # abbreviation expansion reconciles different names.
    if len(original_names) < 2:
        continue

    abbreviation_candidates.append({
        "entity_type":
            group["entity_type"].iloc[0],

        "match_name":
            group["match_name"].iloc[0],

        "canonical_address":
            group["canonical_address"].iloc[0],

        "canonical_names":
            original_names,

        "entity_count":
            len(group),

        "occurrence_count":
            group[
                "occurrence_count"
            ].sum(),

        "row_indexes":
            tuple(
                group.index
            )
    })


abbreviation_candidates = pd.DataFrame(
    abbreviation_candidates
)

print(
    "Common-abbreviation candidate groups:",
    len(abbreviation_candidates)
)

abbreviation_candidates

Common-abbreviation candidate groups: 5


,entity_type,match_name,canonical_address,canonical_names,entity_count,occurrence_count,row_indexes
0,Organization,"ONE DAY, LLC","1417 ACADEMY, KALAMAZOO, MI 49006","(ONE DAY, LLC, ONE DAY, LLC.)",2,4,"(1624, 1643)"
1,Organization,"STD MANAGEMENT, LLC","3001 10TH ST, WAYLAND, MI 49348","(S.T.D. MANAGEMENT, LLC, STD MANAGEMENT, LLC)",2,6,"(1771, 2244)"
2,Organization,J SAJE LLC,"2900 KENSINGTON DR, KALAMAZOO, MI 49008","(J SAJE LLC, J. SAJE LLC)",2,4,"(1802, 4494)"
3,Organization,MAIN QSR INCORPORATED,"1946 45TH ST STE C, MUNSTER, IN 46321","(MAIN QSR INC, MAIN QSR INC.)",2,2,"(4038, 4510)"
4,Organization,AIM CORPORATION,"218 GLADYS ST, PORTAGE, MI 49002","(AIM CORP, AIM CORPORATION)",2,2,"(4240, 4329)"


In [91]:
if not abbreviation_candidates.empty:

    print(
        abbreviation_candidates[
            "entity_type"
        ].value_counts()
    )

entity_type
Organization    5
Name: count, dtype: int64


## Entities with the highest occurrence_count

In [92]:
entities.sort_values(
    "occurrence_count",
    ascending=False
).head(10)

,entity_id,canonical_name,entity_type,canonical_address,organization_type,organization_classification_confidence,occurrence_count
2356,2357,KAYLOR HOLDINGS LLC,Organization,"1405 HOLIDAY LN, PORTAGE, MI 49024",LLC,High,62
661,662,"BROOKS, SAMUEL C",Person,"804 CHARLOTTE AVE, KALAMAZOO, MI 49048",NaN,NaN,51
4292,4293,STATE WIDE RENTALS LLC,Organization,"606 BRYANT ST, KALAMAZOO, MI 49001",LLC,High,25
1310,1311,"DIPZINSKI, JOSHUA D",Person,"13044 MONINGTON CT, FENTON, MI 48430-1191",NaN,NaN,24
20,21,"19 PROPS, LLC",Organization,"200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 49007",LLC,High,21
1701,1702,"GIBSON, ROBERT M",Person,"8713 S 38TH ST, SCOTTS, MI 49088",NaN,NaN,21
1698,1699,"GIBSON, MARTHA M",Person,"8713 S 38TH ST, SCOTTS, MI 49088",NaN,NaN,19
478,479,"BISHOP, BRADLEY J",Person,"PO BOX 312, PORTAGE, MI 49081",NaN,NaN,19
1998,1999,"HERTEL, JOSEPH B",Person,"2326 S ROSE ST, KALAMAZOO, MI 49001",NaN,NaN,18
4759,4760,WALLY BROWN LLC,Organization,"563 TREASURE ISLAND DR, MATTAWAN, MI 49071",LLC,High,18


## Entities with canonical names containing a middle name or middle initial

In [93]:
SUFFIXES = {
    "JR",
    "SR",
    "JUNIOR",
    "SENIOR",
    "II",
    "III",
    "IV",
}


def get_middle_tokens(name):
    """
    Return possible middle-name tokens from a person name.

    Supports both:
        SMITH, JOHN A
        SMITH JOHN A

    This is for manual inspection only, not entity-resolution logic.
    """

    if pd.isna(name):
        return tuple()

    name = str(name).upper().strip()
    name = name.replace(".", "")

    # Skip obvious joint/compound records
    if (
        "&" in name
        or "/" in name
        or re.search(r"\bAND\b", name)
    ):
        return tuple()

    if "," in name:

        _, given_part = name.split(",", 1)
        tokens = given_part.strip().split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # FIRST [MIDDLE...]
        return tuple(tokens[1:])

    else:

        tokens = name.split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # Assume LAST FIRST [MIDDLE...]
        if len(tokens) < 3:
            return tuple()

        return tuple(tokens[2:])

people_with_middle_names = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

people_with_middle_names["middle_tokens"] = (
    people_with_middle_names[
        "canonical_name"
    ]
    .apply(get_middle_tokens)
)

people_with_middle_names = (
    people_with_middle_names[
        people_with_middle_names[
            "middle_tokens"
        ].apply(len) > 0
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

people_with_middle_names[
    [
        "entity_id",
        "canonical_name",
        "middle_tokens",
        "canonical_address",
        "occurrence_count"
    ]
]

,entity_id,canonical_name,middle_tokens,canonical_address,occurrence_count
661,662,"BROOKS, SAMUEL C","(C,)","804 CHARLOTTE AVE, KALAMAZOO, MI 49048",51
1310,1311,"DIPZINSKI, JOSHUA D","(D,)","13044 MONINGTON CT, FENTON, MI 48430-1191",24
1701,1702,"GIBSON, ROBERT M","(M,)","8713 S 38TH ST, SCOTTS, MI 49088",21
478,479,"BISHOP, BRADLEY J","(J,)","PO BOX 312, PORTAGE, MI 49081",19
1698,1699,"GIBSON, MARTHA M","(M,)","8713 S 38TH ST, SCOTTS, MI 49088",19
...,...,...,...,...,...
155,156,"AMBRO, ANDREW J","(J,)","11300 E D AVE, RICHLAND, MI 49083",1
156,157,"AMBRO, ANDREW J","(J,)","1251 W GULL LAKE DR, RICHLAND, MI 49083",1
158,159,"AMBRO, MARILEE K","(K,)","11300 E D AVE, RICHLAND, MI 49083",1
159,160,"AMBRO, MARILEE K","(K,)","1251 W GULL LAKE DR, RICHLAND, MI 49083",1


- Some names have commas while others don't
    - For example, KAVANAUGH, JAMES J vs. CLAWSON DORRINDA L
        - Note that both had their middle initial listed correctly under middle_tokens
- The canonical name FRITH, TERRANCE JEROME A produced the middle_token (JEROME, A)
- The canonical name BROWN, ROBERT WENDELL, JR. produced the middle_token (WENDELL,,)
    - Note that there are two commas rather than the usual one
- GREATER TRUE WORSHIP COGIC produced (WORSHIP, COGIC)
- BLACKSTAR STABILITY REVITALIZATION produced (REVITALIZATION,)
- EDWARDS, KARRIE; EDWARDS, KENTON produced (EDWARDS,, KENTON)

Unrelated case:
There are two entities with the name WIDGER JAMES LEROY, one with the address 214 MARSHALL ST, ALLEGAN, MI 49010 and the other with the address 214 MARSHALL ST, ALLEGAN, MI 49010. Because they have the exact same name and their address, with the removal of ST from the first case, are the same - they should clearly be merged.

### Distinguishing middle initials from full middle names

In [94]:
def middle_type(tokens):

    if not tokens:
        return None

    if all(len(token) == 1 for token in tokens):
        return "Initial"

    return "Full/Other"


people_with_middle_names["middle_type"] = (
    people_with_middle_names[
        "middle_tokens"
    ]
    .apply(middle_type)
)

people_with_middle_names[
    "middle_type"
].value_counts()

middle_type
Initial       1394
Full/Other     397
Name: count, dtype: int64

Out of 3,955 unique entity names, 1,240 show as containing a middle name or initial.

In [95]:
797 + 443

1240

## Entities the previously had a blank address

In [96]:
blank_address_cases = (
    resolution_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ],
        dropna=False
    )
    .agg(
        had_blank_address=(
            "associated_address",
            lambda x: (
                x.fillna("")
                .str.strip()
                .eq("")
                .any()
            )
        ),
        source_addresses=(
            "associated_address",
            lambda x: tuple(
                sorted(
                    {
                        address
                        for address in x.fillna("")
                        if address != ""
                    }
                )
            )
        ),
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        occurrence_count=(
            "occurrence_id",
            "size"
        )
    )
    .reset_index()
)

blank_address_cases = (
    blank_address_cases[
        blank_address_cases["had_blank_address"]
        &
        (
            blank_address_cases[
                "canonical_address"
            ] != ""
        )
    ]
)

print(blank_address_cases.shape)
blank_address_cases.head(50)

(1922, 7)


,entity_id,canonical_name,canonical_address,had_blank_address,source_addresses,source_names,occurrence_count
15,16,1733 W. MAIN ST LLC,"1733 1/2 W MAIN ST, KALAMAZOO, MI 49006",True,"(1733 1/2 W MAIN ST, KALAMAZOO, MI 49006,)","(1733 W. MAIN ST LLC,)",2
19,20,19 PROPS LLC,"200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 49007-",True,"(200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 490...","(19 PROPS LLC,)",2
21,22,"1930 WEST MAIN, LLC","3260 NILES RD STE B, SAINT JOSEPH, MI 49085",True,"(3260 NILES RD STE B, SAINT JOSEPH, MI 49085,)","(1930 WEST MAIN, LLC,)",2
24,25,2219 HUMPHREY LLC,"100 W MICHIGAN AVE STE 200, KALAMAZOO, MI 49007",True,"(100 W MICHIGAN AVE STE 200, KALAMAZOO, MI 490...","(2219 HUMPHREY LLC,)",2
26,27,232 GRAND LLC,"11670 RAMSEYER DR, PLAINWELL, MI 49080",True,"(11670 RAMSEYER DR, PLAINWELL, MI 49080,)","(232 GRAND LLC,)",2
28,29,26TH ST PROPERTIES LLC,"2915 E GRAND RIVER AVE, HOWELL, MI 48843",True,"(2915 E GRAND RIVER AVE, HOWELL, MI 48843,)","(26TH ST PROPERTIES LLC,)",2
38,39,3030 S 9TH LLC,"PO BOX 312098, NEW BRAUNFELS, TX 78131",True,"(PO BOX 312098, NEW BRAUNFELS, TX 78131,)","(3030 S 9TH LLC,)",2
41,42,3381 FOREST VIEW INVESTORS LLC,"2265 LIVERNOIS STE 500, TROY, MI 48083",True,"(2265 LIVERNOIS STE 500, TROY, MI 48083,)","(3381 FOREST VIEW INVESTORS LLC,)",4
42,43,"3501 LAKE HOLDINGS, LLC","3501 LAKE ST, KALAMAZOO, MI 49048",True,"(3501 LAKE ST, KALAMAZOO, MI 49048,)","(3501 LAKE HOLDINGS, LLC,)",2
49,50,5752 TRUST,"4426 HUGH HOWELL RD # B-369, TUCKER, GA 30084",True,"(4426 HUGH HOWELL RD # B-369, TUCKER, GA 30084,)","(5752 TRUST,)",2


There are 1,580 cases of entities that used to have a blank address
- **QUESTION: Is the main thing to look for here that the source_names and canonical_name are in fact similar enough that we would want to merge them?**

## Names affected by CORP/CORPORATION, INC/INCORPORATED, CO/COMPANY

In [97]:
ENTITY_NAME_REPLACEMENTS = {
    "CORP": "CORPORATION",
    "INC": "INCORPORATED",
    "CO": "COMPANY"
}


def expand_entity_abbreviations(name):

    if pd.isna(name):
        return ""

    name = str(name).upper().strip()

    name = name.replace(".", "")

    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"\s*,\s*", ", ", name)

    for old, new in ENTITY_NAME_REPLACEMENTS.items():

        name = re.sub(
            rf"\b{re.escape(old)}\b",
            new,
            name
        )

    return name.strip()

abbreviation_audit = resolution_audit.copy()

abbreviation_audit["expanded_name"] = (
    abbreviation_audit[
        "entity_name"
    ]
    .apply(expand_entity_abbreviations)
)

abbreviation_cases = (
    abbreviation_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        expanded_names=(
            "expanded_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        source_name_count=(
            "entity_name",
            "nunique"
        ),
        expanded_name_count=(
            "expanded_name",
            "nunique"
        )
    )
    .reset_index()
)

abbreviation_cases = (
    abbreviation_cases[
        (abbreviation_cases["source_name_count"] > 1)
        &
        (abbreviation_cases["expanded_name_count"] == 1)
    ]
)

abbreviation_cases.head(50)

,entity_id,canonical_name,canonical_address,source_names,expanded_names,source_name_count,expanded_name_count
113,114,AIM CORPORATION,"218 GLADYS ST, PORTAGE, MI 49002","(AIM CORP, AIM CORPORATION)","(AIM CORPORATION,)",2,1
220,221,"APPS, GARY E.","3503 KENBROOKE CT, KALAMAZOO, MI 49006-5436","(APPS, GARY E, APPS, GARY E.)","(APPS, GARY E,)",2,1
2177,2178,J. SAJE LLC,"2900 KENSINGTON DR, KALAMAZOO, MI 49008","(J SAJE LLC, J. SAJE LLC)","(J SAJE LLC,)",2,1
2755,2756,MAIN QSR INC.,"1946 45TH ST STE C, MUNSTER, IN 46321","(MAIN QSR INC, MAIN QSR INC.)","(MAIN QSR INCORPORATED,)",2,1
3351,3352,"ONE DAY, LLC.","1417 ACADEMY, KALAMAZOO, MI 49006","(ONE DAY, LLC, ONE DAY, LLC.)","(ONE DAY, LLC,)",2,1
3939,3940,"S.T.D. MANAGEMENT, LLC","3001 10TH ST, WAYLAND, MI 49348","(S.T.D. MANAGEMENT, LLC, STD MANAGEMENT, LLC)","(STD MANAGEMENT, LLC,)",2,1


- There is 1 case of a CORP/CORPORATION merger.
- Other than that, the other 5 cases involve merging source names that differ by the inclusion/exclusion of a period.
    - For example, ONE DAY, LLC and ONE DAY, LLC.

## Suffix cases

### All people containing a suffix

In [98]:
suffix_pattern = (
    r"\b(?:JR|SR|JUNIOR|SENIOR|II|III|IV)\b"
)

suffix_entities = (
    entities[
        (entities["entity_type"] == "Person")
        &
        (
            entities["canonical_name"]
            .str.contains(
                suffix_pattern,
                regex=True,
                na=False
            )
        )
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

print(suffix_entities.shape)

suffix_entities[
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
].head(50)

(121, 7)


,entity_id,canonical_name,canonical_address,occurrence_count
175,176,"ANDERSON, A DAVID SR","PO BOX 50204, KALAMAZOO, MI 49005-0204",15
3635,3636,"PRIMER, DANIEL L JR","6841 N SPRINKLE RD, KALAMAZOO, MI 49004",12
2458,2459,"KNOX, DANNY LEE SR","6336 RAVINE RD, KALAMAZOO, MI 49009-9002",9
2197,2198,"JACKSON, SOL JR","5370 LUCERNE AVE, KALAMAZOO, MI 49048",8
1286,1287,"DERRING, ALBERT III","3041 N 26TH ST, KALAMAZOO, MI 49048-4827",7
1206,1207,"DAVIS, KEITH SR.","3850 ROLLRIDGE, KALAMAZOO, MI 49004",5
688,689,"BROWN, ROBERT WENDELL, JR.","1115 DUPONT AVE, KALAMAZOO, MI 49048-3213",5
2053,2054,"HOLLAND SR, MICHAEL","556 CAMANO WAY, MCDONOUGH, GA 30253-8376",5
1756,1757,"GOOCH, MYRON EVERETT JR","1226 CADET LN, KALAMAZOO, MI 49009",4
4752,4753,WALL ROBERT JR,"1057 N 30TH ST, GALESBURG, MI 49053",4


- 146 entities contain suffixes

### Entities with both suffix and suffix-free versions

In [99]:
suffix_audit = (
    resolution_audit[
        resolution_audit[
            "entity_type_final"
        ] == "Person"
    ]
    .copy()
)

suffix_audit["has_suffix"] = (
    suffix_audit[
        "entity_name"
    ]
    .str.contains(
        suffix_pattern,
        regex=True,
        na=False
    )
)

suffix_merge_cases = (
    suffix_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        has_suffix_version=(
            "has_suffix",
            "any"
        ),
        has_no_suffix_version=(
            "has_suffix",
            lambda x: (~x).any()
        )
    )
    .reset_index()
)

suffix_merge_cases = (
    suffix_merge_cases[
        suffix_merge_cases["has_suffix_version"]
        &
        suffix_merge_cases["has_no_suffix_version"]
    ]
)

suffix_merge_cases.head(50)

,entity_id,canonical_name,canonical_address,source_names,has_suffix_version,has_no_suffix_version


- There is 1 case of an entity merged due to having the same name and address, with different suffixes
    - Albert Derring and Albert Derring III

## Cases of finalized entities with the same name but different associated addresses

In [100]:
person_entities = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

known_address_people = (
    person_entities[
        person_entities[
            "canonical_address"
        ].fillna("").str.strip() != ""
    ]
)

common_names_multiple_addresses = (
    known_address_people
    .groupby("canonical_name")
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),
        address_count=(
            "canonical_address",
            "nunique"
        ),
        addresses=(
            "canonical_address",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        entity_ids=(
            "entity_id",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        total_occurrences=(
            "occurrence_count",
            "sum"
        )
    )
    .reset_index()
)

common_names_multiple_addresses = (
    common_names_multiple_addresses[
        common_names_multiple_addresses[
            "address_count"
        ] > 1
    ]
    .sort_values(
        [
            "address_count",
            "total_occurrences"
        ],
        ascending=False
    )
)

print(common_names_multiple_addresses.shape)
common_names_multiple_addresses.head(50)

(332, 6)


,canonical_name,entity_count,address_count,addresses,entity_ids,total_occurrences
2855,"SANTIAGO, JORGE",5,5,"(5101 SWALLOW AVE, KALAMAZOO, MI 49009, 5350 G...","(3985, 3986, 3987, 3988, 3989)",19
2857,"SANTIAGO, MARIA",4,4,"(5101 SWALLOW AVE, KALAMAZOO, MI 49009, 5350 G...","(3994, 3995, 3996, 3997)",18
2428,"OWENS, LESSIE T",4,4,"(1214 N CHURCH ST, KALAMAZOO, MI 49007-3498, 1...","(3394, 3395, 3396, 3397)",9
183,"BALL, RONALD",4,4,"(1004 WOODWARD AVE, KALAMAZOO, MI 49007, 1821 ...","(318, 319, 320, 321)",5
2135,"MEIER, ARTHUR C",3,3,"(1226 CADET LN, KALAMAZOO, MI 49009, 1805 E MA...","(2967, 2968, 2969)",15
425,"BRINNEMAN, MICHAEL J",3,3,"(1802 FULFORD ST, KALAMAZOO, MI 49001, 5037 ST...","(633, 634, 635)",11
3392,"WADE, GEORGE R",3,3,"(1102 S BURDICK ST, KALAMAZOO, MI 49001, 1102 ...","(4729, 4730, 4731)",10
1067,"FAYLING, CHRISTOPHER M",3,3,"(2086 S 36TH ST, GALESBURG, MI 49053, 504 W VA...","(1505, 1506, 1507)",8
3153,"TARDANI, JACOB A",3,3,"(1011 DENNER ST UNIT 1, KALAMAZOO, MI 49006, 1...","(4402, 4403, 4404)",7
3540,"WILLIAMS, LATASHA",3,3,"(1321 CAMERON ST, KALAMAZOO, MI 49001, 1321 CA...","(4926, 4927, 4928)",7


- There are 296 canonical_names which are associated with more than one entity/address

## Inspecting name formatting

In [101]:
people = (
    entity_occurrences[
        entity_occurrences["entity_type"] == "Person"
    ]
)

comma_names = (
    people["entity_name"]
    .str.contains(",", regex=False, na=False)
)

print(
    f"Person occurrences with comma-format names: "
    f"{comma_names.sum():,}"
)

print(
    f"Person occurrences without comma-format names: "
    f"{(~comma_names).sum():,}"
)

print(
    f"Percent parseable by current comma requirement: "
    f"{comma_names.mean():.1%}"
)

Person occurrences with comma-format names: 7,969
Person occurrences without comma-format names: 1,575
Percent parseable by current comma requirement: 83.5%


## Inspecting unknown entity types

In [102]:
entity_occurrences[
    "entity_type"
].value_counts()

entity_type
Person          9544
Organization    2002
Joint            472
Unknown           38
Name: count, dtype: int64

In [103]:
unknown_names = (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Unknown",
        "entity_name"
    ]
    .value_counts()
    .rename_axis("entity_name")
    .reset_index(name="occurrence_count")
)

print(
    f"Unique Unknown names: "
    f"{len(unknown_names):,}"
)

unknown_names.head(100)

Unique Unknown names: 18


,entity_name,occurrence_count
0,"KALE, ANDRIJA-WALDBILLIG, ZORICA",6
1,PAYMENT IN LIEU OF TAXES,4
2,"VANDERMOLEN, JUSTIN-KIECZYNSKI, GR",3
3,"MCNEES, JR",3
4,"CUTLER, IRA, DORIS, STEVEN II, DALE",2
5,"BARNETT R, SABATTIS H, WHARTON L,",2
6,"MI 4004 DOUGLAS AVE. KALAMAZOO, LL",2
7,CENTURYLINK,2
8,"SMITH, LASHALE, SMITH, TEVIN",2
9,TRSTEE,2


In [104]:
joint_names = (
    entity_occurrences.loc[
        entity_occurrences[
            "entity_type"
        ] == "Joint",
        "entity_name"
    ]
    .drop_duplicates()
)

joint_names.sample(
    min(100, len(joint_names)),
    random_state=42
).sort_values()

4511               ALFORD, MICHAEL B & N J
3481      ANDERSON GWEN V & TD, ML, LD, NL
149     ANTISDALE THOMAS & CRANE ELIZABETH
4453            BAKHUYZEN, MICHAEL H & K F
3535       BARTZEN MICHAEL & CRAW ARLENE E
                       ...                
451     WEELDREYER ANDIS JON & ADRIAN LOIS
766     WERTKIN, SUSAN GATES & ROBERT ALAN
4866     WILLIAMSON CHARLES & CLARK NADINE
4812    WILSON KEITH & M & STOCKWELL L & T
7302             WITT DAVID JC & MENDY RAE
Name: entity_name, Length: 100, dtype: str

In [105]:
joint_summary = pd.Series({
    "Contains &": joint_names.str.contains(
        "&",
        regex=False
    ).sum(),

    "Contains AND": joint_names.str.contains(
        r"\bAND\b",
        regex=True
    ).sum(),

    "Contains /": joint_names.str.contains(
        "/",
        regex=False
    ).sum(),

    "Contains comma": joint_names.str.contains(
        ",",
        regex=False
    ).sum(),
})

joint_summary

Contains &        180
Contains AND        7
Contains /         26
Contains comma    101
dtype: int64

## Inspecting Joint type entities

In [106]:
joint_names = (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Joint",
        "entity_name"
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

def profile_joint_name(name):

    candidate = resolver._normalize_entity_type_candidate(name)

    return pd.Series({
        "ampersand_count": candidate.count("&"),
        "semicolon_count": candidate.count(";"),
        "slash_count": candidate.count("/"),

        "and_count": len(
            re.findall(
                r"\bAND\b",
                candidate
            )
        ),

        "comma_count": candidate.count(","),

        "contains_et_al": bool(
            re.search(
                r"\bET AL\b",
                candidate
            )
        ),

        "token_count": len(
            candidate.split()
        )
    })

joint_profile = (
    joint_names
    .to_frame(name="entity_name")
)

joint_profile = pd.concat(
    [
        joint_profile,

        joint_profile["entity_name"]
        .apply(profile_joint_name)
    ],
    axis=1
)

joint_profile.head()

,entity_name,ampersand_count,semicolon_count,slash_count,and_count,comma_count,contains_et_al,token_count
0,CLAWSON DORRINDA &,1,0,0,0,0,False,3
1,SHILTS NANCY & MCCOY THOMAS,1,0,0,0,0,False,5
2,PAK SAM K & KANG IM KYO,1,0,0,0,0,False,7
3,GRAY MARK & MCCOOK MELISSA,1,0,0,0,0,False,5
4,ALEXANDER F & PAVELEC J & PAFF D L,2,0,0,0,0,False,9


In [107]:
joint_profile["separator_type"] = "Other"

joint_profile.loc[
    (
        joint_profile["ampersand_count"] == 1
    )
    &
    (
        joint_profile[
            [
                "semicolon_count",
                "slash_count",
                "and_count"
            ]
        ].sum(axis=1) == 0
    ),
    "separator_type"
] = "Single ampersand"

joint_profile.loc[
    joint_profile["semicolon_count"] > 0,
    "separator_type"
] = "Semicolon"

joint_profile.loc[
    joint_profile["slash_count"] > 0,
    "separator_type"
] = "Slash"

joint_profile.loc[
    joint_profile["and_count"] > 0,
    "separator_type"
] = "AND"

joint_profile.loc[
    (
        joint_profile[
            [
                "ampersand_count",
                "semicolon_count",
                "slash_count",
                "and_count"
            ]
        ].sum(axis=1) > 1
    ),
    "separator_type"
] = "Multiple separators"

joint_profile["separator_type"].value_counts()

separator_type
Single ampersand       152
Multiple separators     31
Slash                   15
AND                      6
Semicolon                3
Other                    2
Name: count, dtype: int64

In [108]:
simple_ampersand = (
    joint_profile[
        (
            joint_profile["separator_type"]
            == "Single ampersand"
        )
        &
        (~joint_profile["contains_et_al"])
    ]
)

print(
    f"Unique simple ampersand Joint names: "
    f"{len(simple_ampersand):,}"
)

simple_ampersand[
    [
        "entity_name",
        "comma_count",
        "token_count"
    ]
].sample(
    min(100, len(simple_ampersand)),
    random_state=42
)

Unique simple ampersand Joint names: 146


,entity_name,comma_count,token_count
57,MICHAEL L. & SARAH D. CANTRELL,0,6
144,FINK JOHN & HUFF MELLISA J,0,6
31,WEELDREYER ANDIS JON & ADRIAN LOIS,0,6
22,GLEASON GEORGE & BOWERS AMY,0,5
52,"TRACY, ELLEN A & RUTH SHEPPARD",1,6
...,...,...,...
20,ROMACK DOUGLAS L & BRADSHAW SUSANNE,0,6
43,"CORONADO, FIDEL & ORTA IVONE",1,5
98,STANFILL NICOLE M & SALTER CHARLES,0,6
174,FOX JOSE & PIER STEPHANIE (LC),0,5


In [109]:
simple_ampersand["comma_count"].value_counts().sort_index()

comma_count
0    81
1    61
2     3
3     1
Name: count, dtype: int64

In [110]:
simple_ampersand.groupby(
    "comma_count"
)["token_count"].describe()

,count,mean,std,min,25%,50%,75%,max
comma_count,,,,,,,,
0,81.0,5.358025,0.884712,3.0,5.0,5.0,6.0,7.0
1,61.0,5.442623,0.940294,3.0,5.0,6.0,6.0,7.0
2,3.0,5.666667,1.154701,5.0,5.0,5.0,6.0,7.0
3,1.0,8.000000,NaN,8.0,8.0,8.0,8.0,8.0


In [111]:
print(len(simple_ampersand))

print(
    simple_ampersand["comma_count"]
    .isna()
    .sum()
)

146
0


In [112]:
medium_joint_candidates = (
    entity_occurrences[
        (entity_occurrences["entity_type"] == "Joint")
        &
        (
            entity_occurrences["joint_parse_confidence"]
            == "Medium"
        )
    ]
    .copy()
)

print(
    "Medium-confidence Joint occurrences:",
    f"{len(medium_joint_candidates):,}"
)

print(
    "Unique Medium-confidence Joint names:",
    f"{medium_joint_candidates['entity_name'].nunique():,}"
)

Medium-confidence Joint occurrences: 0
Unique Medium-confidence Joint names: 0


In [113]:
medium_joint_candidates[
    "joint_parse_rule"
].value_counts()

Series([], Name: count, dtype: int64)

In [114]:
medium_joint_names = (
    medium_joint_candidates[
        [
            "entity_name",
            "joint_parse_rule"
        ]
    ]
    .drop_duplicates()
)

medium_joint_names.sample(
    min(100, len(medium_joint_names)),
    random_state=42
).sort_values(
    "entity_name"
)

,entity_name,joint_parse_rule


In [115]:
expanded_joint_audit = (
    entity_occurrences[
        entity_occurrences["joint_expanded"]
    ]
    [
        [
            "source_occurrence_id",
            "source_entity_name",
            "entity_name",
            "joint_party_index",
            "joint_party_count",
            "joint_parse_rule",
            "joint_parse_confidence"
        ]
    ]
    .sort_values(
        [
            "source_occurrence_id",
            "joint_party_index"
        ]
    )
)

expanded_joint_audit.head(100)

,source_occurrence_id,source_entity_name,entity_name,joint_party_index,joint_party_count,joint_parse_rule,joint_parse_confidence
4,5,SPIEKERMANN TIM & JUDY,"SPIEKERMANN, TIM",1.0,2.0,shared_surname_single_right_name,High
5,5,SPIEKERMANN TIM & JUDY,"SPIEKERMANN, JUDY",2.0,2.0,shared_surname_single_right_name,High
6,6,LINDBERG MARK H & LYNNETTE,"LINDBERG, MARK H",1.0,2.0,shared_surname_single_right_name,High
7,6,LINDBERG MARK H & LYNNETTE,"LINDBERG, LYNNETTE",2.0,2.0,shared_surname_single_right_name,High
18,17,THOMSON STEVEN L & ASHLEY A,"THOMSON, STEVEN L",1.0,2.0,shared_surname_initial_no_comma,Medium
...,...,...,...,...,...,...,...
246,199,"LONEY, MARY & STEVE","LONEY, STEVE",2.0,2.0,shared_surname_comma,High
252,205,"KORTZ, TAYLOR & DZIEWICKI, GREGORY","KORTZ, TAYLOR",1.0,2.0,explicit_surnames_comma,High
253,205,"KORTZ, TAYLOR & DZIEWICKI, GREGORY","DZIEWICKI, GREGORY",2.0,2.0,explicit_surnames_comma,High
255,207,"LECHTANSKI, KEVIN & SANDRA","LECHTANSKI, KEVIN",1.0,2.0,shared_surname_comma,High


In [116]:
expanded_joint_pairs = (
    expanded_joint_audit
    .pivot_table(
        index=[
            "source_occurrence_id",
            "source_entity_name",
            "joint_parse_rule",
            "joint_parse_confidence"
        ],
        columns="joint_party_index",
        values="entity_name",
        aggfunc="first"
    )
    .reset_index()
    .rename(
        columns={
            1: "person_1",
            2: "person_2"
        }
    )
)

medium_joint_names.sort_values(
    "entity_name"
).reset_index(
    drop=True
)

,entity_name,joint_parse_rule


In [117]:
expanded_counts = (
    entity_occurrences[
        entity_occurrences["joint_expanded"]
    ]
    .groupby(
        "source_occurrence_id"
    )
    .size()
)

print(
    "Expanded source occurrences not producing exactly 2 rows:",
    (expanded_counts != 2).sum()
)

Expanded source occurrences not producing exactly 2 rows: 0


In [118]:
expanded_source_ids = set(
    entity_occurrences.loc[
        entity_occurrences["joint_expanded"],
        "source_occurrence_id"
    ]
)

surviving_joint_rows = (
    entity_occurrences[
        entity_occurrences["source_occurrence_id"]
        .isin(expanded_source_ids)
        &
        (
            entity_occurrences["entity_type"]
            == "Joint"
        )
    ]
)

print(
    "Expanded sources still containing a Joint row:",
    len(surviving_joint_rows)
)

Expanded sources still containing a Joint row: 0


In [119]:
entity_occurrences["entity_type"].value_counts()

entity_type
Person          9544
Organization    2002
Joint            472
Unknown           38
Name: count, dtype: int64

## Analyzing Joint type entities after splitting high and medium confidence one and two comma Joint entities

In [120]:
remaining_joint = (
    entity_occurrences[
        entity_occurrences["entity_type"] == "Joint"
    ]
    .copy()
)

print(
    "Remaining Joint occurrences:",
    f"{len(remaining_joint):,}"
)

print(
    "Unique remaining Joint names:",
    f"{remaining_joint['entity_name'].nunique():,}"
)

Remaining Joint occurrences: 472
Unique remaining Joint names: 209


In [121]:
remaining_joint_names = (
    remaining_joint["entity_name"]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [122]:
remaining_parseable = (
    remaining_joint_names
    .apply(
        lambda name:
        resolver._parse_joint_name(name)
        is not None
    )
)

print(
    "Remaining Joint names still parseable "
    "by current parser:",
    remaining_parseable.sum()
)

Remaining Joint names still parseable by current parser: 0


In [123]:
def profile_joint_name(name):

    candidate = (
        resolver
        ._normalize_entity_type_candidate(name)
    )

    return pd.Series({
        "ampersand_count":
            candidate.count("&"),

        "semicolon_count":
            candidate.count(";"),

        "slash_count":
            candidate.count("/"),

        "and_count":
            len(
                re.findall(
                    r"\bAND\b",
                    candidate
                )
            ),

        "comma_count":
            candidate.count(","),

        "contains_et_al":
            bool(
                re.search(
                    r"\bET AL\b",
                    candidate
                )
            ),

        "token_count":
            len(candidate.split())
    })

In [124]:
remaining_joint_profile = (
    remaining_joint_names
    .to_frame(name="entity_name")
)

remaining_joint_profile = pd.concat(
    [
        remaining_joint_profile,

        remaining_joint_profile[
            "entity_name"
        ].apply(
            profile_joint_name
        )
    ],
    axis=1
)

In [125]:
remaining_joint_profile[
    "comma_count"
].value_counts().sort_index()

comma_count
0    108
1     83
2     17
3      1
Name: count, dtype: int64

In [126]:
remaining_joint_profile[
    [
        "ampersand_count",
        "semicolon_count",
        "slash_count",
        "and_count"
    ]
].describe()

,ampersand_count,semicolon_count,slash_count,and_count
count,209.000000,209.000000,209.000000,209.000000
mean,0.990431,0.033493,0.119617,0.033493
std,0.588270,0.205285,0.353618,0.180352
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000
75%,1.000000,0.000000,0.000000,0.000000
max,4.000000,2.000000,2.000000,1.000000


In [127]:
zero_comma_single_ampersand = (
    remaining_joint_profile[
        (remaining_joint_profile["ampersand_count"] == 1)
        &
        (remaining_joint_profile["comma_count"] == 0)
        &
        (remaining_joint_profile["semicolon_count"] == 0)
        &
        (remaining_joint_profile["slash_count"] == 0)
        &
        (remaining_joint_profile["and_count"] == 0)
        &
        (~remaining_joint_profile["contains_et_al"])
    ]
    .copy()
)

print(
    "Unique zero-comma single-ampersand names:",
    f"{len(zero_comma_single_ampersand):,}"
)

Unique zero-comma single-ampersand names: 81


In [128]:
zero_comma_single_ampersand[
    [
        "entity_name",
        "token_count"
    ]
].sample(
    min(
        100,
        len(zero_comma_single_ampersand)
    ),
    random_state=42
).sort_values(
    "entity_name"
)

,entity_name,token_count
173,& FALES SHERRIE,3
184,AARON DANIEL & AABERG GAYE,5
10,AHLRICH JANITH L &,4
110,ALLEN JENNIFER & RICHTER RUSSELL E,6
12,ANTISDALE THOMAS & CRANE ELIZABETH,5
...,...,...
8,VILLANUEVA MIGUEL &,3
21,WADINGTON LOREN A & DAVID L TRSTEES,7
31,WEELDREYER ANDIS JON & ADRIAN LOIS,6
186,WILLIAMSON CHARLES & CLARK NADINE,5


## Analyzing single ampersand cases with no commas

In [129]:
def profile_zero_comma_joint(name):

    candidate = (
        resolver
        ._normalize_entity_type_candidate(name)
    )

    left, right = [
        part.strip()
        for part in candidate.split("&", 1)
    ]

    left_tokens = left.split()
    right_tokens = right.split()

    return pd.Series({
        "left": left,
        "right": right,

        "left_token_count":
            len(left_tokens),

        "right_token_count":
            len(right_tokens),

        "left_empty":
            len(left_tokens) == 0,

        "right_empty":
            len(right_tokens) == 0,

        "left_first_token":
            left_tokens[0]
            if left_tokens
            else "",

        "right_first_token":
            right_tokens[0]
            if right_tokens
            else "",

        "right_last_token":
            right_tokens[-1]
            if right_tokens
            else "",

        "right_ends_initial":
            bool(
                right_tokens
                and len(right_tokens[-1]) == 1
            ),

        "same_leading_token":
            bool(
                left_tokens
                and right_tokens
                and left_tokens[0]
                == right_tokens[0]
            )
    })


zero_comma_structure = pd.concat(
    [
        zero_comma_single_ampersand[
            ["entity_name"]
        ].reset_index(drop=True),

        zero_comma_single_ampersand[
            "entity_name"
        ]
        .reset_index(drop=True)
        .apply(profile_zero_comma_joint)
    ],
    axis=1
)

In [130]:
zero_comma_structure[
    "right_token_count"
].value_counts().sort_index()

right_token_count
0     7
2    58
3    16
Name: count, dtype: int64

In [131]:
pd.crosstab(
    zero_comma_structure[
        "right_token_count"
    ],
    zero_comma_structure[
        "right_ends_initial"
    ]
)

right_ends_initial,False,True
right_token_count,,
0,7,0
2,57,1
3,8,8


In [132]:
print(
    "Trailing-empty records:",
    zero_comma_structure[
        "right_empty"
    ].sum()
)

print(
    "Same leading token on both sides:",
    zero_comma_structure[
        "same_leading_token"
    ].sum()
)

Trailing-empty records: 7
Same leading token on both sides: 0


### Continuing analysis

In [133]:
pd.crosstab(
    zero_comma_structure["left_token_count"],
    zero_comma_structure["right_token_count"]
)

left_one_token = (
    zero_comma_structure[
        (zero_comma_structure["left_token_count"] == 1)
        &
        (~zero_comma_structure["right_empty"])
    ]
    .copy()
)

print(
    "Unique names with one token on the left:",
    len(left_one_token)
)

left_one_token[
    [
        "entity_name",
        "left",
        "right",
        "right_token_count"
    ]
].sort_values(
    "entity_name"
)

Unique names with one token on the left: 1


,entity_name,left,right,right_token_count
74,C/O SCOFIELD & VAN DENBELDT,SCOFIELD,VAN DENBELDT,2


In [134]:
right_two_ending_initial = (
    zero_comma_structure[
        (zero_comma_structure["right_token_count"] == 2)
        &
        (zero_comma_structure["right_ends_initial"])
        &
        (~zero_comma_structure["right_empty"])
    ]
    .copy()
)

right_two_ending_initial[
    [
        "entity_name",
        "left",
        "right",
        "left_token_count"
    ]
].sort_values(
    "entity_name"
)

,entity_name,left,right,left_token_count
34,CAMPBELL J SR & L M,CAMPBELL J SR,L M,3


In [135]:
right_two_full_tokens = (
    zero_comma_structure[
        (zero_comma_structure["right_token_count"] == 2)
        &
        (~zero_comma_structure["right_ends_initial"])
    ]
    .copy()
)

right_two_full_tokens[
    [
        "entity_name",
        "left",
        "right",
        "left_token_count"
    ]
].sort_values(
    "entity_name"
)

,entity_name,left,right,left_token_count
65,& FALES SHERRIE,,FALES SHERRIE,0
71,AARON DANIEL & AABERG GAYE,AARON DANIEL,AABERG GAYE,2
11,ANTISDALE THOMAS & CRANE ELIZABETH,ANTISDALE THOMAS,CRANE ELIZABETH,2
27,BAIRD ROBERT & MARY ANNE,BAIRD ROBERT,MARY ANNE,2
26,BAIRD ROBERT R & MARY ANNE,BAIRD ROBERT R,MARY ANNE,3
21,BELL ERIK & MARBLE MORGAN,BELL ERIK,MARBLE MORGAN,2
67,BENHAM MATHEW & VEGA HEATHER (LC),BENHAM MATHEW,VEGA HEATHER,2
58,BLOK BRENNON & JONGEKRYG ABAGAIL,BLOK BRENNON,JONGEKRYG ABAGAIL,2
42,BRIERLY ROBERT & STREET DANIEL,BRIERLY ROBERT,STREET DANIEL,2
13,BUCHHOLZ ROGER & DELAYNE CHARVAT,BUCHHOLZ ROGER,DELAYNE CHARVAT,2


In [136]:
entity_occurrences.loc[
    entity_occurrences["joint_expanded"],
    "joint_parse_rule"
].value_counts()

joint_parse_rule
shared_surname_comma                   1714
shared_surname_single_right_name        750
shared_surname_initial_no_comma         560
explicit_surnames_comma                 480
explicit_shared_surname_no_comma         54
shared_trailing_surname_no_comma         10
explicit_surnames_initials_no_comma       4
Name: count, dtype: int64

## Investigating building a name dictionary from county data

In [137]:
known_first_names = set()
known_last_names = set()

for name in (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Person",
        "entity_name"
    ]
    .drop_duplicates()
):

    normalized = resolver._normalize_match_name(
        name
    )

    # Only use comma-format names for building the vocabulary.
    # Their name orientation is explicitly represented in the
    # source rather than inferred.
    if "," not in normalized:
        continue

    parsed = resolver._parse_person_name(
        normalized
    )

    if parsed is None:
        continue

    known_first_names.add(
        parsed["first_name"]
    )

    known_last_names.add(
        parsed["last_name"]
    )

In [138]:
def get_name_position_evidence(row):

    right_tokens = row["right"].split()

    if len(right_tokens) != 2:
        return pd.Series({
            "right_token_1": "",
            "right_token_2": "",
            "token_1_seen_first": False,
            "token_1_seen_last": False,
            "token_2_seen_first": False,
            "token_2_seen_last": False
        })

    token_1, token_2 = right_tokens

    return pd.Series({
        "right_token_1": token_1,
        "right_token_2": token_2,

        "token_1_seen_first":
            token_1 in known_first_names,

        "token_1_seen_last":
            token_1 in known_last_names,

        "token_2_seen_first":
            token_2 in known_first_names,

        "token_2_seen_last":
            token_2 in known_last_names
    })


right_two_full_evidence = pd.concat(
    [
        right_two_full_tokens.reset_index(
            drop=True
        ),

        right_two_full_tokens.reset_index(
            drop=True
        ).apply(
            get_name_position_evidence,
            axis=1
        )
    ],
    axis=1
)

In [139]:
def classify_right_side_evidence(row):

    # Strong evidence for:
    #
    # SURNAME FIRST
    #
    # e.g. MCCADEN SHAWNA
    if (
        row["token_1_seen_last"]
        and row["token_2_seen_first"]
        and not row["token_1_seen_first"]
    ):
        return "Likely separate surname"

    # Strong evidence for:
    #
    # FIRST MIDDLE
    #
    # e.g. MARY ANNE
    if (
        row["token_1_seen_first"]
        and not row["token_1_seen_last"]
        and not row["token_2_seen_last"]
    ):
        return "Likely shared surname"

    return "Ambiguous"


right_two_full_evidence[
    "name_evidence"
] = right_two_full_evidence.apply(
    classify_right_side_evidence,
    axis=1
)

In [140]:
right_two_full_evidence[
    "name_evidence"
].value_counts()

name_evidence
Ambiguous                  36
Likely separate surname    11
Likely shared surname      10
Name: count, dtype: int64

In [141]:
right_two_full_evidence[
    [
        "entity_name",
        "right",
        "right_token_1",
        "right_token_2",
        "token_1_seen_first",
        "token_1_seen_last",
        "token_2_seen_first",
        "token_2_seen_last",
        "name_evidence"
    ]
].sort_values(
    [
        "name_evidence",
        "entity_name"
    ]
)

,entity_name,right,right_token_1,right_token_2,token_1_seen_first,token_1_seen_last,token_2_seen_first,token_2_seen_last,name_evidence
42,& FALES SHERRIE,FALES SHERRIE,FALES,SHERRIE,False,False,True,False,Ambiguous
47,AARON DANIEL & AABERG GAYE,AABERG GAYE,AABERG,GAYE,False,False,False,False,Ambiguous
4,ANTISDALE THOMAS & CRANE ELIZABETH,CRANE ELIZABETH,CRANE,ELIZABETH,False,False,True,False,Ambiguous
38,BLOK BRENNON & JONGEKRYG ABAGAIL,JONGEKRYG ABAGAIL,JONGEKRYG,ABAGAIL,False,False,False,False,Ambiguous
27,BRIERLY ROBERT & STREET DANIEL,STREET DANIEL,STREET,DANIEL,False,False,True,True,Ambiguous
6,BUCHHOLZ ROGER & DELAYNE CHARVAT,DELAYNE CHARVAT,DELAYNE,CHARVAT,False,False,False,False,Ambiguous
20,BUSTER WILLIAM & MCCADEN SHAWNA,MCCADEN SHAWNA,MCCADEN,SHAWNA,False,False,False,False,Ambiguous
52,C/O PR DEBRA BRITTEN & STEVEN HUNT,STEVEN HUNT,STEVEN,HUNT,True,False,False,True,Ambiguous
50,C/O SCOFIELD & VAN DENBELDT,VAN DENBELDT,VAN,DENBELDT,False,True,False,False,Ambiguous
46,C/O: HARJINDER KAUR & RAJEET KAUR,RAJEET KAUR,RAJEET,KAUR,False,False,False,True,Ambiguous


### Adding address data

In [142]:
person_evidence_rows = []

for _, row in entity_occurrences.iterrows():

    # Avoid using Joint-derived people to prove our own Joint
    # decomposition assumptions.
    if row["entity_type"] != "Person":
        continue

    if row.get("joint_expanded", False):
        continue

    parsed = resolver._parse_person_name(
        row["entity_name"]
    )

    if parsed is None:
        continue

    person_evidence_rows.append({
        "entity_name": row["entity_name"],
        "associated_address": row["associated_address"],
        "last_name": parsed["last_name"],
        "first_name": parsed["first_name"],
        "middle_tokens": parsed["middle_tokens"],
        "suffix": parsed["suffix"]
    })


person_evidence = pd.DataFrame(
    person_evidence_rows
)

In [143]:
def evaluate_two_full_token_joint(row):

    left_tokens = row["left"].split()
    right_tokens = row["right"].split()

    result = {
        "shared_candidate": "",
        "separate_candidate": "",
        "shared_exact_anywhere": False,
        "separate_exact_anywhere": False,
        "shared_same_address": False,
        "separate_same_address": False
    }

    if (
        len(left_tokens) < 2
        or len(right_tokens) != 2
    ):
        return pd.Series(result)

    shared_surname = left_tokens[0]

    right_1 = right_tokens[0]
    right_2 = right_tokens[1]

    # Hypothesis A:
    # right side is FIRST MIDDLE and inherits left surname.
    shared_candidate = (
        f"{shared_surname}, "
        f"{right_1} {right_2}"
    )

    # Hypothesis B:
    # right side is LAST FIRST.
    separate_candidate = (
        f"{right_1}, "
        f"{right_2}"
    )

    result[
        "shared_candidate"
    ] = shared_candidate

    result[
        "separate_candidate"
    ] = separate_candidate

    shared_parsed = (
        resolver._parse_person_name(
            shared_candidate
        )
    )

    separate_parsed = (
        resolver._parse_person_name(
            separate_candidate
        )
    )

    address = row.get(
        "associated_address",
        ""
    )

    if shared_parsed is not None:

        shared_matches = person_evidence[
            (person_evidence["last_name"]
             == shared_parsed["last_name"])
            &
            (person_evidence["first_name"]
             == shared_parsed["first_name"])
            &
            (person_evidence["middle_tokens"]
             == shared_parsed["middle_tokens"])
        ]

        result[
            "shared_exact_anywhere"
        ] = len(shared_matches) > 0

        result[
            "shared_same_address"
        ] = (
            (
                shared_matches[
                    "associated_address"
                ] == address
            ).any()
            if address
            else False
        )

    if separate_parsed is not None:

        separate_matches = person_evidence[
            (person_evidence["last_name"]
             == separate_parsed["last_name"])
            &
            (person_evidence["first_name"]
             == separate_parsed["first_name"])
            &
            (person_evidence["middle_tokens"]
             == separate_parsed["middle_tokens"])
        ]

        result[
            "separate_exact_anywhere"
        ] = len(separate_matches) > 0

        result[
            "separate_same_address"
        ] = (
            (
                separate_matches[
                    "associated_address"
                ] == address
            ).any()
            if address
            else False
        )

    return pd.Series(result)

In [144]:
right_two_full_with_address = (
    right_two_full_tokens
    .merge(
        remaining_joint[
            [
                "entity_name",
                "associated_address"
            ]
        ].drop_duplicates(),
        on="entity_name",
        how="left"
    )
)

right_two_full_with_address

,entity_name,left,right,left_token_count,right_token_count,left_empty,right_empty,left_first_token,right_first_token,right_last_token,right_ends_initial,same_leading_token,associated_address
0,SHILTS NANCY & MCCOY THOMAS,SHILTS NANCY,MCCOY THOMAS,2,2,False,False,SHILTS,MCCOY,THOMAS,False,False,"8881 W C AVE, KALAMAZOO, MI 49009"
1,SHILTS NANCY & MCCOY THOMAS,SHILTS NANCY,MCCOY THOMAS,2,2,False,False,SHILTS,MCCOY,THOMAS,False,False,
2,GRAY MARK & MCCOOK MELISSA,GRAY MARK,MCCOOK MELISSA,2,2,False,False,GRAY,MCCOOK,MELISSA,False,False,"6708 OWEN DR, KALAMAZOO, MI 49009"
3,GRAY MARK & MCCOOK MELISSA,GRAY MARK,MCCOOK MELISSA,2,2,False,False,GRAY,MCCOOK,MELISSA,False,False,
4,PLOWMAN GLENN W & JUDITH ETA,PLOWMAN GLENN W,JUDITH ETA,3,2,False,False,PLOWMAN,JUDITH,ETA,False,False,"3504 W D AVE, KALAMAZOO, MI 49009"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,C/O PR DEBRA BRITTEN & STEVEN HUNT,PR DEBRA BRITTEN,STEVEN HUNT,3,2,False,False,PR,STEVEN,HUNT,False,False,"1369 M DR S, E LEROY, MI 49051"
92,PLOWMAN GLENN W & JUDITH ET,PLOWMAN GLENN W,JUDITH ET,3,2,False,False,PLOWMAN,JUDITH,ET,False,False,"3504 W D AVE, KALAMAZOO, MI 49009"
93,GATES MARIAN & KELLOGG HARR,GATES MARIAN,KELLOGG HARR,2,2,False,False,GATES,KELLOGG,HARR,False,False,"5461 COLLINGWOOD, KALAMAZOO, MI 49004"
94,NEFF JR KENNETH & FAUBLE TIFFANY,NEFF JR KENNETH,FAUBLE TIFFANY,3,2,False,False,NEFF,FAUBLE,TIFFANY,False,False,"6233 E W AVE, VICKSBURG, MI 49097"


In [145]:
context_evidence = pd.concat(
    [
        right_two_full_with_address
        .reset_index(drop=True),

        right_two_full_with_address
        .reset_index(drop=True)
        .apply(
            evaluate_two_full_token_joint,
            axis=1
        )
    ],
    axis=1
)

context_evidence

,entity_name,left,right,left_token_count,right_token_count,left_empty,right_empty,left_first_token,right_first_token,right_last_token,right_ends_initial,same_leading_token,associated_address,shared_candidate,separate_candidate,shared_exact_anywhere,separate_exact_anywhere,shared_same_address,separate_same_address
0,SHILTS NANCY & MCCOY THOMAS,SHILTS NANCY,MCCOY THOMAS,2,2,False,False,SHILTS,MCCOY,THOMAS,False,False,"8881 W C AVE, KALAMAZOO, MI 49009","SHILTS, MCCOY THOMAS","MCCOY, THOMAS",False,False,False,False
1,SHILTS NANCY & MCCOY THOMAS,SHILTS NANCY,MCCOY THOMAS,2,2,False,False,SHILTS,MCCOY,THOMAS,False,False,,"SHILTS, MCCOY THOMAS","MCCOY, THOMAS",False,False,False,False
2,GRAY MARK & MCCOOK MELISSA,GRAY MARK,MCCOOK MELISSA,2,2,False,False,GRAY,MCCOOK,MELISSA,False,False,"6708 OWEN DR, KALAMAZOO, MI 49009","GRAY, MCCOOK MELISSA","MCCOOK, MELISSA",False,False,False,False
3,GRAY MARK & MCCOOK MELISSA,GRAY MARK,MCCOOK MELISSA,2,2,False,False,GRAY,MCCOOK,MELISSA,False,False,,"GRAY, MCCOOK MELISSA","MCCOOK, MELISSA",False,False,False,False
4,PLOWMAN GLENN W & JUDITH ETA,PLOWMAN GLENN W,JUDITH ETA,3,2,False,False,PLOWMAN,JUDITH,ETA,False,False,"3504 W D AVE, KALAMAZOO, MI 49009","PLOWMAN, JUDITH ETA","JUDITH, ETA",False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,C/O PR DEBRA BRITTEN & STEVEN HUNT,PR DEBRA BRITTEN,STEVEN HUNT,3,2,False,False,PR,STEVEN,HUNT,False,False,"1369 M DR S, E LEROY, MI 49051","PR, STEVEN HUNT","STEVEN, HUNT",False,False,False,False
92,PLOWMAN GLENN W & JUDITH ET,PLOWMAN GLENN W,JUDITH ET,3,2,False,False,PLOWMAN,JUDITH,ET,False,False,"3504 W D AVE, KALAMAZOO, MI 49009","PLOWMAN, JUDITH ET","JUDITH, ET",False,False,False,False
93,GATES MARIAN & KELLOGG HARR,GATES MARIAN,KELLOGG HARR,2,2,False,False,GATES,KELLOGG,HARR,False,False,"5461 COLLINGWOOD, KALAMAZOO, MI 49004","GATES, KELLOGG HARR","KELLOGG, HARR",False,False,False,False
94,NEFF JR KENNETH & FAUBLE TIFFANY,NEFF JR KENNETH,FAUBLE TIFFANY,3,2,False,False,NEFF,FAUBLE,TIFFANY,False,False,"6233 E W AVE, VICKSBURG, MI 49097","NEFF, FAUBLE TIFFANY","FAUBLE, TIFFANY",False,False,False,False


In [146]:
context_evidence_by_name = (
    context_evidence
    .groupby(
        "entity_name",
        as_index=False
    )
    .agg(
        shared_candidate=(
            "shared_candidate",
            "first"
        ),
        separate_candidate=(
            "separate_candidate",
            "first"
        ),

        shared_exact_anywhere=(
            "shared_exact_anywhere",
            "any"
        ),
        separate_exact_anywhere=(
            "separate_exact_anywhere",
            "any"
        ),

        shared_same_address=(
            "shared_same_address",
            "any"
        ),
        separate_same_address=(
            "separate_same_address",
            "any"
        ),

        address_count=(
            "associated_address",
            "nunique"
        )
    )
)

In [147]:
print(
    "Unique Joint names in contextual analysis:",
    len(context_evidence_by_name)
)

Unique Joint names in contextual analysis: 57


In [148]:
pd.crosstab(
    context_evidence_by_name[
        "shared_same_address"
    ],
    context_evidence_by_name[
        "separate_same_address"
    ],
    margins=True
)

separate_same_address,False,All
shared_same_address,,
False,57,57
All,57,57


In [149]:
same_address_evidence = (
    context_evidence_by_name[
        context_evidence_by_name[
            [
                "shared_same_address",
                "separate_same_address"
            ]
        ].any(axis=1)
    ]
    [
        [
            "entity_name",
            "shared_candidate",
            "separate_candidate",
            "shared_same_address",
            "separate_same_address",
            "shared_exact_anywhere",
            "separate_exact_anywhere",
            "address_count"
        ]
    ]
    .sort_values(
        [
            "shared_same_address",
            "separate_same_address",
            "entity_name"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
)

same_address_evidence

,entity_name,shared_candidate,separate_candidate,shared_same_address,separate_same_address,shared_exact_anywhere,separate_exact_anywhere,address_count


In [150]:
def classify_contextual_evidence(row):

    shared_same = row[
        "shared_same_address"
    ]

    separate_same = row[
        "separate_same_address"
    ]

    shared_any = row[
        "shared_exact_anywhere"
    ]

    separate_any = row[
        "separate_exact_anywhere"
    ]


    # ------------------------------------------------------
    # Strongest evidence: same address
    # ------------------------------------------------------

    if shared_same and not separate_same:
        return "Shared surname - same address"

    if separate_same and not shared_same:
        return "Separate surnames - same address"

    if shared_same and separate_same:
        return "Conflict - both same address"


    # ------------------------------------------------------
    # Secondary evidence: exact candidate elsewhere
    # ------------------------------------------------------

    if shared_any and not separate_any:
        return "Shared surname - elsewhere"

    if separate_any and not shared_any:
        return "Separate surnames - elsewhere"

    if shared_any and separate_any:
        return "Conflict - both elsewhere"


    # ------------------------------------------------------
    # No contextual evidence
    # ------------------------------------------------------

    return "No contextual evidence"

In [151]:
context_evidence_by_name[
    "context_decision"
] = (
    context_evidence_by_name
    .apply(
        classify_contextual_evidence,
        axis=1
    )
)

In [152]:
context_evidence_by_name[
    "context_decision"
].value_counts()

context_decision
No contextual evidence    57
Name: count, dtype: int64

## Addressing cases of same name similar addresses

In [153]:
STREET_SUFFIXES = {
    "ST",
    "AVE",
    "RD",
    "DR",
    "BLVD",
    "LN",
    "CT",
    "CIR",
    "PL",
    "PKWY",
    "TER",
    "HWY",
    "WAY",
    "TRL"
}

UNIT_MARKERS = {
    "APT",
    "STE",
    "UNIT",
    "BLDG",
    "FL",
    "RM"
}


def parse_address_suffix_for_audit(address):
    """
    Break a cleaned address into components needed to detect
    cases where a street suffix is present in one representation
    and absent in another.

    This is for analysis only and does not modify the address.
    """

    empty_result = pd.Series({
        "street_base": None,
        "street_suffix": None,
        "unit": None,
        "locality": None
    })

    if pd.isna(address):
        return empty_result

    address = str(address).strip()

    if not address:
        return empty_result

    parts = [
        part.strip()
        for part in address.split(",")
    ]

    # Require at least:
    #
    # street, locality...
    if len(parts) < 2:
        return empty_result

    street_line = parts[0]

    # Don't try to interpret PO Boxes as street addresses.
    if street_line.startswith("PO BOX"):
        return empty_result

    locality = ", ".join(parts[1:])

    tokens = street_line.split()

    if len(tokens) < 2:
        return empty_result


    # ----------------------------------------------------------
    # Separate unit information
    # ----------------------------------------------------------

    unit_index = None

    for i, token in enumerate(tokens):

        if (
            token in UNIT_MARKERS
            or token.startswith("#")
        ):
            unit_index = i
            break

    if unit_index is not None:

        street_tokens = tokens[:unit_index]
        unit_tokens = tokens[unit_index:]

    else:

        street_tokens = tokens
        unit_tokens = []


    if len(street_tokens) < 2:
        return empty_result


    # Require something resembling a street number.
    if not any(
        character.isdigit()
        for character in street_tokens[0]
    ):
        return empty_result


    # ----------------------------------------------------------
    # Identify street suffix
    # ----------------------------------------------------------

    if street_tokens[-1] in STREET_SUFFIXES:

        street_suffix = street_tokens[-1]

        street_base_tokens = (
            street_tokens[:-1]
        )

    else:

        street_suffix = ""

        street_base_tokens = street_tokens


    if not street_base_tokens:
        return empty_result


    return pd.Series({
        "street_base":
            " ".join(street_base_tokens),

        "street_suffix":
            street_suffix,

        "unit":
            " ".join(unit_tokens),

        "locality":
            locality
    })

In [154]:
address_suffix_audit = (
    entities[
        entities["canonical_address"]
        .fillna("")
        .str.strip()
        .ne("")
    ]
    .copy()
)

parsed_addresses = (
    address_suffix_audit[
        "canonical_address"
    ]
    .apply(
        parse_address_suffix_for_audit
    )
)

address_suffix_audit = pd.concat(
    [
        address_suffix_audit.reset_index(
            drop=True
        ),

        parsed_addresses.reset_index(
            drop=True
        )
    ],
    axis=1
)

In [155]:
address_suffix_candidates = []

group_columns = [
    "entity_type",
    "canonical_name",
    "street_base",
    "unit",
    "locality"
]

for _, group in (
    address_suffix_audit
    .dropna(
        subset=[
            "street_base",
            "street_suffix",
            "locality"
        ]
    )
    .groupby(
        group_columns,
        dropna=False,
        sort=False
    )
):

    if len(group) < 2:
        continue

    suffixes = set(
        group["street_suffix"]
    )

    nonblank_suffixes = {
        suffix
        for suffix in suffixes
        if suffix
    }

    # Require:
    #
    #   one suffix-free version
    #   +
    #   exactly one known suffix
    #
    # Example:
    #
    #   214 MARSHALL
    #   214 MARSHALL ST
    #
    if (
        "" not in suffixes
        or len(nonblank_suffixes) != 1
    ):
        continue

    address_suffix_candidates.append({
        "entity_type":
            group["entity_type"].iloc[0],

        "canonical_name":
            group["canonical_name"].iloc[0],

        "street_base":
            group["street_base"].iloc[0],

        "locality":
            group["locality"].iloc[0],

        "unit":
            group["unit"].iloc[0],

        "known_suffix":
            next(iter(nonblank_suffixes)),

        "entity_ids":
            tuple(
                group["entity_id"]
            ),

        "addresses":
            tuple(
                group["canonical_address"]
            ),

        "entity_count":
            len(group)
    })


address_suffix_candidates = pd.DataFrame(
    address_suffix_candidates
)

In [156]:
print(
    "Candidate missing-street-suffix groups:",
    len(address_suffix_candidates)
)

address_suffix_candidates

Candidate missing-street-suffix groups: 0


""


In [157]:
entities["occurrence_count"].sum()

np.int64(12056)

In [158]:
entities[
    entities["canonical_name"]
    .str.contains(
        "WIDGER JAMES LEROY",
        regex=False,
        na=False
    )
]

,entity_id,canonical_name,entity_type,canonical_address,organization_type,organization_classification_confidence,occurrence_count
4895,4896,WIDGER JAMES LEROY,Person,"214 MARSHALL ST, ALLEGAN, MI 49010",NaN,NaN,6


In [159]:
entities.head()
entities.info()

<class 'pandas.DataFrame'>
RangeIndex: 5078 entries, 0 to 5077
Data columns (total 7 columns):
 #   Column                                  Non-Null Count  Dtype
---  ------                                  --------------  -----
 0   entity_id                               5078 non-null   int64
 1   canonical_name                          5078 non-null   str  
 2   entity_type                             5078 non-null   str  
 3   canonical_address                       5078 non-null   str  
 4   organization_type                       777 non-null    str  
 5   organization_classification_confidence  777 non-null    str  
 6   occurrence_count                        5078 non-null   int64
dtypes: int64(2), str(5)
memory usage: 277.8 KB


In [160]:
print(
    "Unique source occurrences:",
    entity_occurrences["source_occurrence_id"].nunique()
)

print(
    "Entity occurrences after Joint expansion:",
    len(entity_occurrences)
)

print(
    "Unique occurrence IDs:",
    entity_occurrences["occurrence_id"].nunique()
)

print(
    "Sum of final entity occurrence counts:",
    entities["occurrence_count"].sum()
)

Unique source occurrences: 10270
Entity occurrences after Joint expansion: 12056
Unique occurrence IDs: 12056
Sum of final entity occurrence counts: 12056


## Suffix merge change analysis

In [161]:
entities[
    entities["canonical_name"]
    .str.contains(
        "DERRING",
        regex=False,
        na=False
    )
][
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
]

,entity_id,canonical_name,canonical_address,occurrence_count
116,117,ALBERT DERRING,"1908 N EDWARDS, KALAMAZOO, MI 49007",2
117,118,ALBERT DERRING,"PO BOX 2116, KALAMAZOO, MI 49003-2116",1
1285,1286,"DERRING, ALBERT","3041 N 26TH ST, KALAMAZOO, MI 49048",3
1286,1287,"DERRING, ALBERT III","3041 N 26TH ST, KALAMAZOO, MI 49048-4827",7


In [162]:
entities[
    entities["canonical_name"]
    .str.contains(
        "LINDSEY, WILLIAM B",
        regex=False,
        na=False
    )
][
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
]

,entity_id,canonical_name,canonical_address,occurrence_count
2652,2653,"LINDSEY, WILLIAM B","804 HARRISON ST, KALAMAZOO, MI 49007",4
2653,2654,"LINDSEY, WILLIAM B JR","804 HARRISON ST, KALAMAZOO, MI 49007",2


In [163]:
entities["occurrence_count"].sum()
# 12058

np.int64(12056)

## Address zip code auditing

In [164]:
import re


def parse_zip_variant_for_audit(address):
    """
    Extract the address body and ZIP components for auditing
    5-digit ZIP versus ZIP+4 variants.

    Does not modify the original address.
    """

    if pd.isna(address):
        return pd.Series({
            "address_without_zip": None,
            "zip5": None,
            "zip4": None
        })

    address = str(address).strip()

    if not address:
        return pd.Series({
            "address_without_zip": None,
            "zip5": None,
            "zip4": None
        })

    match = re.search(
        r"^(.*\D)(\d{5})(?:-(\d{4}))?$",
        address
    )

    if match is None:
        return pd.Series({
            "address_without_zip": None,
            "zip5": None,
            "zip4": None
        })

    return pd.Series({
        "address_without_zip":
            match.group(1).strip(),

        "zip5":
            match.group(2),

        "zip4":
            match.group(3) or ""
    })

In [165]:
zip_audit = (
    entities[
        entities["canonical_address"]
        .fillna("")
        .str.strip()
        .ne("")
    ]
    .copy()
    .reset_index(drop=True)
)

zip_parts = (
    zip_audit["canonical_address"]
    .apply(parse_zip_variant_for_audit)
)

zip_audit = pd.concat(
    [
        zip_audit,
        zip_parts
    ],
    axis=1
)

In [166]:
zip_variant_candidates = []

group_columns = [
    "entity_type",
    "canonical_name",
    "address_without_zip",
    "zip5"
]

for _, group in (
    zip_audit
    .dropna(
        subset=[
            "address_without_zip",
            "zip5"
        ]
    )
    .groupby(
        group_columns,
        dropna=False,
        sort=False
    )
):

    if len(group) < 2:
        continue

    zip4_values = set(
        group["zip4"]
    )

    nonblank_zip4 = {
        value
        for value in zip4_values
        if value
    }

    # Require:
    #
    #     49048
    #     49048-4827
    #
    # but reject a group containing conflicting ZIP+4 values.
    if (
        "" not in zip4_values
        or len(nonblank_zip4) != 1
    ):
        continue

    zip_variant_candidates.append({
        "entity_type":
            group["entity_type"].iloc[0],

        "canonical_name":
            group["canonical_name"].iloc[0],

        "address_without_zip":
            group["address_without_zip"].iloc[0],

        "zip5":
            group["zip5"].iloc[0],

        "known_zip4":
            next(iter(nonblank_zip4)),

        "addresses":
            tuple(
                group["canonical_address"]
                .drop_duplicates()
            ),

        "entity_ids":
            tuple(
                group["entity_id"]
            ),

        "entity_count":
            len(group)
    })


zip_variant_candidates = pd.DataFrame(
    zip_variant_candidates
)

print(
    "Candidate ZIP+4 variant groups:",
    len(zip_variant_candidates)
)

zip_variant_candidates

Candidate ZIP+4 variant groups: 0


""


## Testing collapse_index_groups change

In [167]:
test_entities = pd.DataFrame({
    "canonical_name": [
        "SMITH, JOHN",
        "SMITH, JOHN A",
        "SMITH, JOHN ALAN"
    ],

    "entity_type": [
        "Person",
        "Person",
        "Person"
    ],

    "canonical_address": [
        "123 MAIN ST, KALAMAZOO, MI 49001",
        "123 MAIN ST, KALAMAZOO, MI 49001",
        "123 MAIN ST, KALAMAZOO, MI 49001"
    ],

    "organization_type": [
        None,
        None,
        None
    ],

    "organization_classification_confidence": [
        None,
        None,
        None
    ],

    "occurrence_count": [
        1,
        1,
        1
    ],

    "_occurrence_ids": [
        (1,),
        (2,),
        (3,)
    ]
})